In [1]:
import json
import platform
import re
import sys
from pathlib import Path

import pandas as pd


PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SAMPLE_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed_rework_v2"
    / "01_5_Eligible_Test_Sample_7500.csv"
)

SAMPLE_METADATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed_rework_v2"
    / "01_5_Eligible_Test_Sample_7500_metadata.json"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed_rework_v2"
)

EXPECTED_LANGUAGES = (
    "go",
    "java",
    "javascript",
    "php",
    "python",
    "ruby",
)

EXPECTED_PER_LANGUAGE = 1250
EXPECTED_TOTAL = 7500


def count_nonempty_lines(text):
    return sum(
        bool(line.strip())
        for line in str(text).splitlines()
    )


def count_words(text):
    return len(
        re.findall(
            r"\b[\w'-]+\b",
            str(text),
            flags=re.UNICODE,
        )
    )


assert SAMPLE_CSV_PATH.exists(), SAMPLE_CSV_PATH
assert SAMPLE_METADATA_PATH.exists(), SAMPLE_METADATA_PATH

sample = pd.read_csv(
    SAMPLE_CSV_PATH,
    keep_default_na=False,
)

with SAMPLE_METADATA_PATH.open(
    mode="r",
    encoding="utf-8",
) as handle:
    sample_metadata = json.load(handle)


required_columns = {
    "sample_id",
    "family_id",
    "language",
    "repository",
    "original_split",
    "original_code",
    "original_comment",
    "comment_line_count",
    "pair_token_count_capped",
    "exact_code_hash",
    "exact_comment_hash",
    "exact_pair_hash",
}

missing_columns = (
    required_columns
    - set(sample.columns)
)

assert not missing_columns, (
    f"Missing required columns: "
    f"{sorted(missing_columns)}"
)

assert len(sample) == EXPECTED_TOTAL
assert sample["family_id"].nunique() == EXPECTED_TOTAL
assert sample["sample_id"].nunique() == EXPECTED_TOTAL
assert set(sample["language"]) == set(EXPECTED_LANGUAGES)
assert (sample["original_split"] == "test").all()
assert sample["original_code"].str.strip().ne("").all()
assert sample["original_comment"].str.strip().ne("").all()
assert sample["pair_token_count_capped"].le(512).all()

language_counts = (
    sample["language"]
    .value_counts()
    .reindex(EXPECTED_LANGUAGES)
)

assert (
    language_counts
    == EXPECTED_PER_LANGUAGE
).all()

assert (
    sample_metadata["sample_size_per_language"]
    == EXPECTED_PER_LANGUAGE
)

assert (
    sample_metadata["total_sample_size"]
    == EXPECTED_TOTAL
)


sample[
    "derived_nonempty_comment_lines"
] = sample["original_comment"].map(
    count_nonempty_lines
)

sample[
    "derived_comment_word_count"
] = sample["original_comment"].map(
    count_words
)

sample[
    "comment_line_count_matches"
] = (
    sample["comment_line_count"]
    == sample[
        "derived_nonempty_comment_lines"
    ]
)


language_summary = (
    sample
    .groupby(
        "language",
        as_index=False,
    )
    .agg(
        records=(
            "family_id",
            "count",
        ),
        repositories=(
            "repository",
            "nunique",
        ),
        minimum_comment_lines=(
            "derived_nonempty_comment_lines",
            "min",
        ),
        median_comment_lines=(
            "derived_nonempty_comment_lines",
            "median",
        ),
        maximum_comment_lines=(
            "derived_nonempty_comment_lines",
            "max",
        ),
        minimum_comment_words=(
            "derived_comment_word_count",
            "min",
        ),
        median_comment_words=(
            "derived_comment_word_count",
            "median",
        ),
        maximum_comment_words=(
            "derived_comment_word_count",
            "max",
        ),
    )
)


line_measurement_check = (
    sample
    .groupby(
        "language",
        as_index=False,
    )
    .agg(
        records=(
            "family_id",
            "count",
        ),
        matching_line_counts=(
            "comment_line_count_matches",
            "sum",
        ),
    )
)

line_measurement_check[
    "matching_percent"
] = (
    100
    * line_measurement_check[
        "matching_line_counts"
    ]
    / line_measurement_check[
        "records"
    ]
).round(2)


print("Environment")
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
print("Project root:", PROJECT_ROOT)

print("\nLoaded sample")
print("Records:", f"{len(sample):,}")
print("Repositories:", sample["repository"].nunique())
print("Languages:", ", ".join(EXPECTED_LANGUAGES))

print("\nLanguage and comment structure")
display(language_summary)

print("\nStored versus derived comment-line measurement")
display(line_measurement_check)

mismatches = sample.loc[
    ~sample["comment_line_count_matches"],
    [
        "sample_id",
        "language",
        "comment_line_count",
        "derived_nonempty_comment_lines",
        "original_comment",
    ],
].head(10)

if mismatches.empty:
    print(
        "\nAll stored comment-line counts match "
        "the non-empty lines derived from the text."
    )
else:
    print("\nExample line-count disagreements")
    display(mismatches)

Environment
Python: 3.13.11
Platform: Windows-11-10.0.26200-SP0
Project root: C:\Users\HP\Desktop\thesis_preprocessing

Loaded sample
Records: 7,500
Repositories: 3204
Languages: go, java, javascript, php, python, ruby

Language and comment structure


,language,records,repositories,minimum_comment_lines,median_comment_lines,maximum_comment_lines,minimum_comment_words,median_comment_words,maximum_comment_words
0,go,1250,211,1,1.0,27,1,12.0,170
1,java,1250,234,1,3.0,37,0,20.0,195
2,javascript,1250,790,1,2.0,26,0,13.0,179
3,php,1250,1029,1,3.0,25,0,13.0,144
4,python,1250,633,1,2.0,18,0,14.0,108
5,ruby,1250,307,1,2.0,37,0,16.0,222



Stored versus derived comment-line measurement


,language,records,matching_line_counts,matching_percent
0,go,1250,1239,99.12
1,java,1250,590,47.20
2,javascript,1250,918,73.44
3,php,1250,335,26.80
4,python,1250,801,64.08
5,ruby,1250,854,68.32



Example line-count disagreements


,sample_id,language,comment_line_count,derived_nonempty_comment_lines,original_comment
39,GO-0040,go,11,10,/* RFC 7208:\n\n As described at the end of ...
349,GO-0350,go,6,5,/*\nOpen returns a new Writer at the specified...
425,GO-0426,go,21,19,"/*\nfunc (r *Render) HTML(status int, name str..."
467,GO-0468,go,6,4,/*\n\nFileSequence\n\n*/\n//export FileSequenc...
486,GO-0487,go,6,5,/*\nRead implements the io.Reader interface. I...
529,GO-0530,go,6,5,/*\nWrite p bytes to our file.\n\nIf our file ...
574,GO-0575,go,5,4,/*\nGetApps gets all app names\n\nGet a list o...
605,GO-0606,go,17,15,/*\nRFC 7208:\n\n4.6.4. DNS Lookup Limits\n\n...
664,GO-0665,go,5,4,/*\nfunc (ipvs *IpvsClient) modifyFWMDest(meth...
736,GO-0737,go,5,4,/*\nGetAppsAppRoutesRoute gets route by name\n...


In [2]:
from datetime import datetime, timezone


EDA_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "rework_v2"
    / "02_distractor_generation"
)

EDA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


def count_physical_splitlines(text):
    lines = str(text).splitlines()
    return max(len(lines), 1)


def count_newline_separated_lines(text):
    return str(text).count("\n") + 1


def count_nonempty_comment_lines(text):
    return sum(
        bool(line.strip())
        for line in str(text).splitlines()
    )


def strip_comment_syntax(line):
    cleaned = str(line)

    cleaned = re.sub(
        r"^\s*(?://+|/\*+|\*+|#+|<!--)\s*",
        "",
        cleaned,
    )

    cleaned = re.sub(
        r"\s*(?:\*/|-->)\s*$",
        "",
        cleaned,
    )

    cleaned = cleaned.strip()

    if cleaned in {
        '"""',
        "'''",
        '"',
        "'",
    }:
        return ""

    return cleaned


def count_content_lines(text):
    return sum(
        bool(strip_comment_syntax(line))
        for line in str(text).splitlines()
    )


line_audit = sample[
    [
        "sample_id",
        "family_id",
        "language",
        "repository",
        "comment_line_count",
        "original_comment",
    ]
].copy()

line_audit[
    "physical_splitlines"
] = line_audit[
    "original_comment"
].map(
    count_physical_splitlines
)

line_audit[
    "newline_separated_lines"
] = line_audit[
    "original_comment"
].map(
    count_newline_separated_lines
)

line_audit[
    "nonempty_lines"
] = line_audit[
    "original_comment"
].map(
    count_nonempty_comment_lines
)

line_audit[
    "content_lines_after_marker_removal"
] = line_audit[
    "original_comment"
].map(
    count_content_lines
)


candidate_definitions = [
    "physical_splitlines",
    "newline_separated_lines",
    "nonempty_lines",
    "content_lines_after_marker_removal",
]

for definition in candidate_definitions:
    line_audit[
        f"{definition}_matches"
    ] = (
        line_audit[definition]
        == line_audit["comment_line_count"]
    )

    line_audit[
        f"{definition}_absolute_difference"
    ] = (
        line_audit[definition]
        - line_audit["comment_line_count"]
    ).abs()


summary_rows = []

for language_value, language_frame in (
    line_audit.groupby(
        "language",
        sort=True,
    )
):
    for definition in candidate_definitions:
        summary_rows.append(
            {
                "language": language_value,
                "definition": definition,
                "records": len(language_frame),
                "exact_matches": int(
                    language_frame[
                        f"{definition}_matches"
                    ].sum()
                ),
                "match_percent": round(
                    100
                    * language_frame[
                        f"{definition}_matches"
                    ].mean(),
                    2,
                ),
                "mean_absolute_difference": round(
                    language_frame[
                        f"{definition}_absolute_difference"
                    ].mean(),
                    3,
                ),
                "maximum_absolute_difference": int(
                    language_frame[
                        f"{definition}_absolute_difference"
                    ].max()
                ),
            }
        )


definition_summary = pd.DataFrame(
    summary_rows
)

overall_rows = []

for definition in candidate_definitions:
    overall_rows.append(
        {
            "definition": definition,
            "records": len(line_audit),
            "exact_matches": int(
                line_audit[
                    f"{definition}_matches"
                ].sum()
            ),
            "match_percent": round(
                100
                * line_audit[
                    f"{definition}_matches"
                ].mean(),
                2,
            ),
            "mean_absolute_difference": round(
                line_audit[
                    f"{definition}_absolute_difference"
                ].mean(),
                3,
            ),
            "maximum_absolute_difference": int(
                line_audit[
                    f"{definition}_absolute_difference"
                ].max()
            ),
        }
    )


overall_summary = (
    pd.DataFrame(overall_rows)
    .sort_values(
        [
            "match_percent",
            "mean_absolute_difference",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)

best_definition = overall_summary.iloc[0][
    "definition"
]


best_definition_mismatches = (
    line_audit.loc[
        ~line_audit[
            f"{best_definition}_matches"
        ],
        [
            "sample_id",
            "language",
            "comment_line_count",
            best_definition,
            "physical_splitlines",
            "newline_separated_lines",
            "nonempty_lines",
            "content_lines_after_marker_removal",
            "original_comment",
        ],
    ]
    .sort_values(
        [
            "language",
            "sample_id",
        ]
    )
)


line_audit_path = (
    EDA_DIR
    / "comment_line_definition_audit.csv"
)

definition_summary_path = (
    EDA_DIR
    / "comment_line_definition_by_language.csv"
)

overall_summary_path = (
    EDA_DIR
    / "comment_line_definition_overall.csv"
)

mismatch_path = (
    EDA_DIR
    / "comment_line_definition_mismatches.csv"
)

manifest_path = (
    EDA_DIR
    / "comment_line_definition_manifest.json"
)


line_audit.to_csv(
    line_audit_path,
    index=False,
    encoding="utf-8",
)

definition_summary.to_csv(
    definition_summary_path,
    index=False,
    encoding="utf-8",
)

overall_summary.to_csv(
    overall_summary_path,
    index=False,
    encoding="utf-8",
)

best_definition_mismatches.to_csv(
    mismatch_path,
    index=False,
    encoding="utf-8",
)


manifest = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_sample": str(
        SAMPLE_CSV_PATH
    ),
    "records": int(
        len(line_audit)
    ),
    "candidate_definitions": (
        candidate_definitions
    ),
    "best_matching_definition": (
        best_definition
    ),
    "best_match_percent": float(
        overall_summary.iloc[0][
            "match_percent"
        ]
    ),
    "mismatch_records": int(
        len(best_definition_mismatches)
    ),
    "output_files": {
        "record_audit": str(
            line_audit_path
        ),
        "language_summary": str(
            definition_summary_path
        ),
        "overall_summary": str(
            overall_summary_path
        ),
        "mismatches": str(
            mismatch_path
        ),
    },
}

with manifest_path.open(
    mode="w",
    encoding="utf-8",
) as handle:
    json.dump(
        manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


print("Overall definition comparison")
display(overall_summary)

print("\nBest definition by language")
display(
    definition_summary.loc[
        definition_summary[
            "definition"
        ] == best_definition
    ].reset_index(drop=True)
)

print(
    "\nBest overall definition:",
    best_definition,
)

print(
    "Unmatched records:",
    f"{len(best_definition_mismatches):,}",
)

if not best_definition_mismatches.empty:
    print("\nExample remaining mismatches")
    display(
        best_definition_mismatches.head(12)
    )

print("\nSaved exploratory outputs")
print("Record audit:", line_audit_path)
print("Language summary:", definition_summary_path)
print("Overall summary:", overall_summary_path)
print("Mismatches:", mismatch_path)
print("Manifest:", manifest_path)

Overall definition comparison


,definition,records,exact_matches,match_percent,mean_absolute_difference,maximum_absolute_difference
0,physical_splitlines,7500,7500,100.00,0.000,0
1,newline_separated_lines,7500,7500,100.00,0.000,0
2,nonempty_lines,7500,4737,63.16,0.557,12
3,content_lines_after_marker_removal,7500,4569,60.92,0.593,12



Best definition by language


,language,definition,records,exact_matches,match_percent,mean_absolute_difference,maximum_absolute_difference
0,go,physical_splitlines,1250,1250,100.0,0.0,0
1,java,physical_splitlines,1250,1250,100.0,0.0,0
2,javascript,physical_splitlines,1250,1250,100.0,0.0,0
3,php,physical_splitlines,1250,1250,100.0,0.0,0
4,python,physical_splitlines,1250,1250,100.0,0.0,0
5,ruby,physical_splitlines,1250,1250,100.0,0.0,0



Best overall definition: physical_splitlines
Unmatched records: 0

Saved exploratory outputs
Record audit: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\comment_line_definition_audit.csv
Language summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\comment_line_definition_by_language.csv
Overall summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\comment_line_definition_overall.csv
Mismatches: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\comment_line_definition_mismatches.csv
Manifest: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\comment_line_definition_manifest.json


In [3]:
from collections import Counter


PERTURBATION_AUDIT_PATH = (
    EDA_DIR
    / "perturbation_structural_feasibility_audit.csv"
)

PERTURBATION_SUMMARY_PATH = (
    EDA_DIR
    / "perturbation_structural_feasibility_by_language.csv"
)

PERTURBATION_GROUP_PATH = (
    EDA_DIR
    / "topic_swap_structural_group_summary.csv"
)


def surface_tokens(text):
    return re.findall(
        r"\S+",
        str(text),
        flags=re.UNICODE,
    )


def line_surface_token_counts(text):
    return [
        len(surface_tokens(line))
        for line in str(text).splitlines()
    ]


perturbation_audit = sample[
    [
        "sample_id",
        "family_id",
        "language",
        "repository",
        "original_comment",
        "comment_line_count",
    ]
].copy()

perturbation_audit[
    "physical_line_count"
] = perturbation_audit[
    "original_comment"
].map(
    count_physical_splitlines
)

perturbation_audit[
    "line_surface_token_counts"
] = perturbation_audit[
    "original_comment"
].map(
    line_surface_token_counts
)

perturbation_audit[
    "total_surface_tokens"
] = perturbation_audit[
    "line_surface_token_counts"
].map(sum)

perturbation_audit[
    "nonempty_surface_token_lines"
] = perturbation_audit[
    "line_surface_token_counts"
].map(
    lambda counts: sum(
        count > 0
        for count in counts
    )
)

perturbation_audit[
    "lines_with_at_least_two_tokens"
] = perturbation_audit[
    "line_surface_token_counts"
].map(
    lambda counts: sum(
        count >= 2
        for count in counts
    )
)

perturbation_audit[
    "lines_with_at_least_three_tokens"
] = perturbation_audit[
    "line_surface_token_counts"
].map(
    lambda counts: sum(
        count >= 3
        for count in counts
    )
)

perturbation_audit[
    "unique_casefolded_surface_tokens"
] = perturbation_audit[
    "original_comment"
].map(
    lambda text: len(
        {
            token.casefold()
            for token in surface_tokens(text)
        }
    )
)

perturbation_audit[
    "shuffle_minimum_eligible"
] = (
    (
        perturbation_audit[
            "total_surface_tokens"
        ] >= 3
    )
    & (
        perturbation_audit[
            "unique_casefolded_surface_tokens"
        ] >= 2
    )
)

assert (
    perturbation_audit[
        "physical_line_count"
    ]
    == perturbation_audit[
        "comment_line_count"
    ]
).all()


line_group_columns = [
    "language",
    "physical_line_count",
]

line_word_group_columns = [
    "language",
    "physical_line_count",
    "total_surface_tokens",
]


perturbation_audit[
    "same_language_line_group_size"
] = (
    perturbation_audit
    .groupby(
        line_group_columns
    )["family_id"]
    .transform("size")
)

perturbation_audit[
    "same_repository_line_group_size"
] = (
    perturbation_audit
    .groupby(
        line_group_columns
        + ["repository"]
    )["family_id"]
    .transform("size")
)

perturbation_audit[
    "different_repository_exact_line_candidates"
] = (
    perturbation_audit[
        "same_language_line_group_size"
    ]
    - perturbation_audit[
        "same_repository_line_group_size"
    ]
)

perturbation_audit[
    "topic_swap_exact_line_eligible"
] = (
    perturbation_audit[
        "different_repository_exact_line_candidates"
    ] > 0
)


perturbation_audit[
    "same_language_line_word_group_size"
] = (
    perturbation_audit
    .groupby(
        line_word_group_columns
    )["family_id"]
    .transform("size")
)

perturbation_audit[
    "same_repository_line_word_group_size"
] = (
    perturbation_audit
    .groupby(
        line_word_group_columns
        + ["repository"]
    )["family_id"]
    .transform("size")
)

perturbation_audit[
    "different_repository_exact_line_word_candidates"
] = (
    perturbation_audit[
        "same_language_line_word_group_size"
    ]
    - perturbation_audit[
        "same_repository_line_word_group_size"
    ]
)

perturbation_audit[
    "topic_swap_exact_line_word_eligible"
] = (
    perturbation_audit[
        "different_repository_exact_line_word_candidates"
    ] > 0
)


perturbation_summary = (
    perturbation_audit
    .groupby(
        "language",
        as_index=False,
    )
    .agg(
        records=(
            "family_id",
            "count",
        ),
        median_surface_tokens=(
            "total_surface_tokens",
            "median",
        ),
        minimum_surface_tokens=(
            "total_surface_tokens",
            "min",
        ),
        maximum_surface_tokens=(
            "total_surface_tokens",
            "max",
        ),
        shuffle_minimum_eligible=(
            "shuffle_minimum_eligible",
            "sum",
        ),
        topic_swap_exact_line_eligible=(
            "topic_swap_exact_line_eligible",
            "sum",
        ),
        topic_swap_exact_line_word_eligible=(
            "topic_swap_exact_line_word_eligible",
            "sum",
        ),
        median_exact_line_candidates=(
            "different_repository_exact_line_candidates",
            "median",
        ),
        median_exact_line_word_candidates=(
            "different_repository_exact_line_word_candidates",
            "median",
        ),
    )
)

for column in [
    "shuffle_minimum_eligible",
    "topic_swap_exact_line_eligible",
    "topic_swap_exact_line_word_eligible",
]:
    perturbation_summary[
        f"{column}_percent"
    ] = (
        100
        * perturbation_summary[column]
        / perturbation_summary["records"]
    ).round(2)


topic_swap_group_summary = (
    perturbation_audit
    .groupby(
        [
            "language",
            "physical_line_count",
            "total_surface_tokens",
        ],
        as_index=False,
    )
    .agg(
        records=(
            "family_id",
            "count",
        ),
        repositories=(
            "repository",
            "nunique",
        ),
    )
    .sort_values(
        [
            "language",
            "physical_line_count",
            "total_surface_tokens",
        ]
    )
)


export_audit = perturbation_audit.copy()

export_audit[
    "line_surface_token_counts"
] = export_audit[
    "line_surface_token_counts"
].map(
    json.dumps
)

export_audit.to_csv(
    PERTURBATION_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

perturbation_summary.to_csv(
    PERTURBATION_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)

topic_swap_group_summary.to_csv(
    PERTURBATION_GROUP_PATH,
    index=False,
    encoding="utf-8",
)


print("Perturbation structural feasibility")
display(
    perturbation_summary[
        [
            "language",
            "records",
            "minimum_surface_tokens",
            "median_surface_tokens",
            "maximum_surface_tokens",
            "shuffle_minimum_eligible",
            "shuffle_minimum_eligible_percent",
            "topic_swap_exact_line_eligible",
            "topic_swap_exact_line_eligible_percent",
            "topic_swap_exact_line_word_eligible",
            "topic_swap_exact_line_word_eligible_percent",
            "median_exact_line_candidates",
            "median_exact_line_word_candidates",
        ]
    ]
)

print("\nOverall feasibility")

overall_feasibility = pd.DataFrame(
    {
        "criterion": [
            "Shuffle minimum eligibility",
            "Topic swap: exact physical lines",
            (
                "Topic swap: exact physical lines "
                "and exact surface-token count"
            ),
        ],
        "eligible_records": [
            int(
                perturbation_audit[
                    "shuffle_minimum_eligible"
                ].sum()
            ),
            int(
                perturbation_audit[
                    "topic_swap_exact_line_eligible"
                ].sum()
            ),
            int(
                perturbation_audit[
                    "topic_swap_exact_line_word_eligible"
                ].sum()
            ),
        ],
    }
)

overall_feasibility[
    "eligible_percent"
] = (
    100
    * overall_feasibility[
        "eligible_records"
    ]
    / len(perturbation_audit)
).round(2)

display(overall_feasibility)

print("\nSaved exploratory outputs")
print("Record audit:", PERTURBATION_AUDIT_PATH)
print("Language summary:", PERTURBATION_SUMMARY_PATH)
print("Structural groups:", PERTURBATION_GROUP_PATH)

Perturbation structural feasibility


,language,records,minimum_surface_tokens,median_surface_tokens,maximum_surface_tokens,shuffle_minimum_eligible,shuffle_minimum_eligible_percent,topic_swap_exact_line_eligible,topic_swap_exact_line_eligible_percent,topic_swap_exact_line_word_eligible,topic_swap_exact_line_word_eligible_percent,median_exact_line_candidates,median_exact_line_word_candidates
0,go,1250,2,13.0,176,1240,99.20,1244,99.52,1119,89.52,697.0,28.0
1,java,1250,1,20.0,183,1156,92.48,1242,99.36,1020,81.60,159.0,4.0
2,javascript,1250,1,13.0,169,1110,88.80,1247,99.76,1041,83.28,113.0,6.0
3,php,1250,1,12.0,136,1140,91.20,1246,99.68,1113,89.04,196.0,8.0
4,python,1250,1,13.0,107,1219,97.52,1247,99.76,1083,86.64,143.0,9.0
5,ruby,1250,1,15.0,227,1149,91.92,1240,99.20,993,79.44,191.0,5.5



Overall feasibility


,criterion,eligible_records,eligible_percent
0,Shuffle minimum eligibility,7014,93.52
1,Topic swap: exact physical lines,7466,99.55
2,Topic swap: exact physical lines and exact sur...,6369,84.92



Saved exploratory outputs
Record audit: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\perturbation_structural_feasibility_audit.csv
Language summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\perturbation_structural_feasibility_by_language.csv
Structural groups: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\topic_swap_structural_group_summary.csv


In [4]:
import math
from bisect import bisect_left, bisect_right
from collections import defaultdict


CALIPER_AUDIT_PATH = (
    EDA_DIR
    / "topic_swap_token_caliper_audit.csv"
)

CALIPER_LANGUAGE_SUMMARY_PATH = (
    EDA_DIR
    / "topic_swap_token_caliper_by_language.csv"
)

CALIPER_OVERALL_SUMMARY_PATH = (
    EDA_DIR
    / "topic_swap_token_caliper_overall.csv"
)


def exact_caliper(token_count):
    return token_count, token_count


def absolute_one_caliper(token_count):
    return (
        max(0, token_count - 1),
        token_count + 1,
    )


def absolute_two_caliper(token_count):
    return (
        max(0, token_count - 2),
        token_count + 2,
    )


def hybrid_ten_percent_caliper(token_count):
    tolerance = max(
        2,
        math.ceil(token_count * 0.10),
    )

    return (
        max(0, token_count - tolerance),
        token_count + tolerance,
    )


def hybrid_fifteen_percent_caliper(token_count):
    tolerance = max(
        3,
        math.ceil(token_count * 0.15),
    )

    return (
        max(0, token_count - tolerance),
        token_count + tolerance,
    )


CALIPER_RULES = {
    "exact_token_count": exact_caliper,
    "absolute_plus_minus_1": absolute_one_caliper,
    "absolute_plus_minus_2": absolute_two_caliper,
    "hybrid_10_percent_minimum_2": (
        hybrid_ten_percent_caliper
    ),
    "hybrid_15_percent_minimum_3": (
        hybrid_fifteen_percent_caliper
    ),
}


group_token_lists = defaultdict(list)
repository_group_token_lists = defaultdict(list)

for row in perturbation_audit.itertuples(
    index=False
):
    group_key = (
        row.language,
        int(row.physical_line_count),
    )

    repository_key = (
        row.language,
        int(row.physical_line_count),
        row.repository,
    )

    group_token_lists[group_key].append(
        int(row.total_surface_tokens)
    )

    repository_group_token_lists[
        repository_key
    ].append(
        int(row.total_surface_tokens)
    )


for token_list in group_token_lists.values():
    token_list.sort()

for token_list in (
    repository_group_token_lists.values()
):
    token_list.sort()


caliper_audit = perturbation_audit[
    [
        "sample_id",
        "family_id",
        "language",
        "repository",
        "physical_line_count",
        "total_surface_tokens",
    ]
].copy()


for rule_name, rule_function in (
    CALIPER_RULES.items()
):
    candidate_counts = []

    for row in caliper_audit.itertuples(
        index=False
    ):
        lower_bound, upper_bound = (
            rule_function(
                int(row.total_surface_tokens)
            )
        )

        group_key = (
            row.language,
            int(row.physical_line_count),
        )

        repository_key = (
            row.language,
            int(row.physical_line_count),
            row.repository,
        )

        all_tokens = group_token_lists[
            group_key
        ]

        repository_tokens = (
            repository_group_token_lists[
                repository_key
            ]
        )

        all_count = (
            bisect_right(
                all_tokens,
                upper_bound,
            )
            - bisect_left(
                all_tokens,
                lower_bound,
            )
        )

        repository_count = (
            bisect_right(
                repository_tokens,
                upper_bound,
            )
            - bisect_left(
                repository_tokens,
                lower_bound,
            )
        )

        candidate_counts.append(
            all_count - repository_count
        )

    caliper_audit[
        f"{rule_name}_candidate_count"
    ] = candidate_counts

    caliper_audit[
        f"{rule_name}_eligible"
    ] = (
        caliper_audit[
            f"{rule_name}_candidate_count"
        ] > 0
    )


language_rows = []

for language_value, language_frame in (
    caliper_audit.groupby(
        "language",
        sort=True,
    )
):
    for rule_name in CALIPER_RULES:
        count_column = (
            f"{rule_name}_candidate_count"
        )

        eligible_column = (
            f"{rule_name}_eligible"
        )

        eligible_counts = language_frame.loc[
            language_frame[
                eligible_column
            ],
            count_column,
        ]

        language_rows.append(
            {
                "language": language_value,
                "caliper_rule": rule_name,
                "records": len(language_frame),
                "eligible_records": int(
                    language_frame[
                        eligible_column
                    ].sum()
                ),
                "eligible_percent": round(
                    100
                    * language_frame[
                        eligible_column
                    ].mean(),
                    2,
                ),
                "median_candidate_count": (
                    float(
                        eligible_counts.median()
                    )
                    if not eligible_counts.empty
                    else 0.0
                ),
                "p10_candidate_count": (
                    float(
                        eligible_counts.quantile(
                            0.10
                        )
                    )
                    if not eligible_counts.empty
                    else 0.0
                ),
                "minimum_candidate_count": (
                    int(
                        eligible_counts.min()
                    )
                    if not eligible_counts.empty
                    else 0
                ),
            }
        )


caliper_language_summary = pd.DataFrame(
    language_rows
)


overall_rows = []

for rule_name in CALIPER_RULES:
    count_column = (
        f"{rule_name}_candidate_count"
    )

    eligible_column = (
        f"{rule_name}_eligible"
    )

    eligible_counts = caliper_audit.loc[
        caliper_audit[
            eligible_column
        ],
        count_column,
    ]

    language_rule_rows = (
        caliper_language_summary.loc[
            caliper_language_summary[
                "caliper_rule"
            ] == rule_name
        ]
    )

    overall_rows.append(
        {
            "caliper_rule": rule_name,
            "eligible_records": int(
                caliper_audit[
                    eligible_column
                ].sum()
            ),
            "eligible_percent": round(
                100
                * caliper_audit[
                    eligible_column
                ].mean(),
                2,
            ),
            "lowest_language_eligibility_percent": float(
                language_rule_rows[
                    "eligible_percent"
                ].min()
            ),
            "median_candidate_count": float(
                eligible_counts.median()
            ),
            "p10_candidate_count": float(
                eligible_counts.quantile(
                    0.10
                )
            ),
        }
    )


caliper_overall_summary = (
    pd.DataFrame(overall_rows)
    .sort_values(
        [
            "eligible_percent",
            "median_candidate_count",
        ],
        ascending=[
            True,
            True,
        ],
    )
    .reset_index(drop=True)
)


caliper_audit.to_csv(
    CALIPER_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

caliper_language_summary.to_csv(
    CALIPER_LANGUAGE_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)

caliper_overall_summary.to_csv(
    CALIPER_OVERALL_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)


print("Overall topic-swap token-caliper comparison")
display(caliper_overall_summary)

print("\nEligibility by language and caliper")
display(
    caliper_language_summary[
        [
            "language",
            "caliper_rule",
            "eligible_records",
            "eligible_percent",
            "median_candidate_count",
            "p10_candidate_count",
        ]
    ]
)

print("\nSaved exploratory outputs")
print("Record audit:", CALIPER_AUDIT_PATH)
print(
    "Language summary:",
    CALIPER_LANGUAGE_SUMMARY_PATH,
)
print(
    "Overall summary:",
    CALIPER_OVERALL_SUMMARY_PATH,
)

Overall topic-swap token-caliper comparison


,caliper_rule,eligible_records,eligible_percent,lowest_language_eligibility_percent,median_candidate_count,p10_candidate_count
0,exact_token_count,6369,84.92,79.44,12.0,1.0
1,absolute_plus_minus_1,6914,92.19,88.56,27.0,3.0
2,absolute_plus_minus_2,7077,94.36,91.68,43.0,4.0
3,hybrid_10_percent_minimum_2,7278,97.04,95.04,43.0,5.0
4,hybrid_15_percent_minimum_3,7331,97.75,96.64,62.0,7.0



Eligibility by language and caliper


,language,caliper_rule,eligible_records,eligible_percent,median_candidate_count,p10_candidate_count
0,go,exact_token_count,1119,89.52,36.0,5.0
1,go,absolute_plus_minus_1,1167,93.36,86.0,7.0
2,go,absolute_plus_minus_2,1195,95.60,142.0,10.0
3,go,hybrid_10_percent_minimum_2,1215,97.20,141.0,12.0
4,go,hybrid_15_percent_minimum_3,1226,98.08,207.0,15.0
5,java,exact_token_count,1020,81.60,6.0,1.0
6,java,absolute_plus_minus_1,1138,91.04,14.0,2.0
7,java,absolute_plus_minus_2,1173,93.84,23.0,3.0
8,java,hybrid_10_percent_minimum_2,1218,97.44,27.0,5.0
9,java,hybrid_15_percent_minimum_3,1222,97.76,36.0,7.0



Saved exploratory outputs
Record audit: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\topic_swap_token_caliper_audit.csv
Language summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\topic_swap_token_caliper_by_language.csv
Overall summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\topic_swap_token_caliper_overall.csv


In [5]:
import hashlib
from collections import defaultdict


TOPIC_SWAP_TOKEN_RULE = (
    "hybrid_10_percent_minimum_2"
)

LEXICAL_THRESHOLDS = (
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
)

LEXICAL_AUDIT_PATH = (
    EDA_DIR
    / "topic_swap_lexical_overlap_audit.csv"
)

LEXICAL_LANGUAGE_SUMMARY_PATH = (
    EDA_DIR
    / "topic_swap_lexical_overlap_by_language.csv"
)

LEXICAL_OVERALL_SUMMARY_PATH = (
    EDA_DIR
    / "topic_swap_lexical_overlap_overall.csv"
)


def normalized_lexical_tokens(text):
    return frozenset(
        token.casefold()
        for token in re.findall(
            r"\w+",
            str(text),
            flags=re.UNICODE,
        )
        if token.strip("_")
    )


def unigram_jaccard(left_tokens, right_tokens):
    union = left_tokens | right_tokens

    if not union:
        return 1.0

    return len(
        left_tokens & right_tokens
    ) / len(union)


def deterministic_pair_rank(
    target_family_id,
    donor_family_id,
):
    value = (
        f"20260805\x1f"
        f"{target_family_id}\x1f"
        f"{donor_family_id}"
    )

    return int.from_bytes(
        hashlib.sha256(
            value.encode("utf-8")
        ).digest()[:8],
        byteorder="big",
        signed=False,
    )


lexical_frame = (
    perturbation_audit.merge(
        sample[
            [
                "family_id",
                "exact_comment_hash",
            ]
        ],
        on="family_id",
        how="left",
        validate="one_to_one",
    )
    .copy()
)

lexical_frame[
    "lexical_token_set"
] = lexical_frame[
    "original_comment"
].map(
    normalized_lexical_tokens
)

lexical_frame[
    "normalized_lexical_token_count"
] = lexical_frame[
    "lexical_token_set"
].map(len)


candidate_groups = defaultdict(list)

for row in lexical_frame.itertuples(
    index=False
):
    candidate_groups[
        (
            row.language,
            int(row.physical_line_count),
        )
    ].append(row)


audit_rows = []

for target in lexical_frame.itertuples(
    index=False
):
    lower_bound, upper_bound = (
        hybrid_ten_percent_caliper(
            int(target.total_surface_tokens)
        )
    )

    group_key = (
        target.language,
        int(target.physical_line_count),
    )

    candidate_results = []

    for donor in candidate_groups[
        group_key
    ]:
        if donor.repository == target.repository:
            continue

        if (
            donor.exact_comment_hash
            == target.exact_comment_hash
        ):
            continue

        donor_token_count = int(
            donor.total_surface_tokens
        )

        if not (
            lower_bound
            <= donor_token_count
            <= upper_bound
        ):
            continue

        jaccard = unigram_jaccard(
            target.lexical_token_set,
            donor.lexical_token_set,
        )

        candidate_results.append(
            (
                jaccard,
                deterministic_pair_rank(
                    target.family_id,
                    donor.family_id,
                ),
                donor.family_id,
                donor.sample_id,
                donor.repository,
                donor.total_surface_tokens,
            )
        )

    candidate_results.sort(
        key=lambda item: (
            item[0],
            item[1],
        )
    )

    result = {
        "sample_id": target.sample_id,
        "family_id": target.family_id,
        "language": target.language,
        "repository": target.repository,
        "physical_line_count": int(
            target.physical_line_count
        ),
        "total_surface_tokens": int(
            target.total_surface_tokens
        ),
        "normalized_lexical_token_count": int(
            target.normalized_lexical_token_count
        ),
        "structural_candidate_count": len(
            candidate_results
        ),
        "minimum_unigram_jaccard": (
            candidate_results[0][0]
            if candidate_results
            else None
        ),
        "best_donor_family_id": (
            candidate_results[0][2]
            if candidate_results
            else None
        ),
        "best_donor_sample_id": (
            candidate_results[0][3]
            if candidate_results
            else None
        ),
        "best_donor_repository": (
            candidate_results[0][4]
            if candidate_results
            else None
        ),
        "best_donor_surface_tokens": (
            candidate_results[0][5]
            if candidate_results
            else None
        ),
    }

    for threshold in LEXICAL_THRESHOLDS:
        threshold_name = str(
            threshold
        ).replace(".", "_")

        qualifying = sum(
            jaccard <= threshold
            for (
                jaccard,
                *_,
            ) in candidate_results
        )

        result[
            f"candidates_at_most_{threshold_name}"
        ] = qualifying

        result[
            f"eligible_at_most_{threshold_name}"
        ] = qualifying > 0

    audit_rows.append(result)


lexical_audit = pd.DataFrame(
    audit_rows
)


language_rows = []

for language_value, language_data in (
    lexical_audit.groupby(
        "language",
        sort=True,
    )
):
    for threshold in LEXICAL_THRESHOLDS:
        threshold_name = str(
            threshold
        ).replace(".", "_")

        count_column = (
            f"candidates_at_most_{threshold_name}"
        )

        eligible_column = (
            f"eligible_at_most_{threshold_name}"
        )

        eligible_counts = language_data.loc[
            language_data[
                eligible_column
            ],
            count_column,
        ]

        language_rows.append(
            {
                "language": language_value,
                "maximum_unigram_jaccard": (
                    threshold
                ),
                "records": len(
                    language_data
                ),
                "eligible_records": int(
                    language_data[
                        eligible_column
                    ].sum()
                ),
                "eligible_percent": round(
                    100
                    * language_data[
                        eligible_column
                    ].mean(),
                    2,
                ),
                "median_qualifying_donors": (
                    float(
                        eligible_counts.median()
                    )
                    if not eligible_counts.empty
                    else 0.0
                ),
                "p10_qualifying_donors": (
                    float(
                        eligible_counts.quantile(
                            0.10
                        )
                    )
                    if not eligible_counts.empty
                    else 0.0
                ),
            }
        )


lexical_language_summary = pd.DataFrame(
    language_rows
)


overall_rows = []

for threshold in LEXICAL_THRESHOLDS:
    threshold_name = str(
        threshold
    ).replace(".", "_")

    count_column = (
        f"candidates_at_most_{threshold_name}"
    )

    eligible_column = (
        f"eligible_at_most_{threshold_name}"
    )

    eligible_counts = lexical_audit.loc[
        lexical_audit[
            eligible_column
        ],
        count_column,
    ]

    threshold_languages = (
        lexical_language_summary.loc[
            lexical_language_summary[
                "maximum_unigram_jaccard"
            ] == threshold
        ]
    )

    overall_rows.append(
        {
            "maximum_unigram_jaccard": threshold,
            "eligible_records": int(
                lexical_audit[
                    eligible_column
                ].sum()
            ),
            "eligible_percent": round(
                100
                * lexical_audit[
                    eligible_column
                ].mean(),
                2,
            ),
            "lowest_language_eligibility_percent": float(
                threshold_languages[
                    "eligible_percent"
                ].min()
            ),
            "median_qualifying_donors": (
                float(
                    eligible_counts.median()
                )
                if not eligible_counts.empty
                else 0.0
            ),
            "p10_qualifying_donors": (
                float(
                    eligible_counts.quantile(
                        0.10
                    )
                )
                if not eligible_counts.empty
                else 0.0
            ),
        }
    )


lexical_overall_summary = pd.DataFrame(
    overall_rows
)


lexical_audit.to_csv(
    LEXICAL_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

lexical_language_summary.to_csv(
    LEXICAL_LANGUAGE_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)

lexical_overall_summary.to_csv(
    LEXICAL_OVERALL_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)


print(
    "Overall lexical-threshold feasibility"
)
display(
    lexical_overall_summary
)

print(
    "\nLexical-threshold feasibility by language"
)
display(
    lexical_language_summary
)

print("\nSaved exploratory outputs")
print("Record audit:", LEXICAL_AUDIT_PATH)
print(
    "Language summary:",
    LEXICAL_LANGUAGE_SUMMARY_PATH,
)
print(
    "Overall summary:",
    LEXICAL_OVERALL_SUMMARY_PATH,
)

Overall lexical-threshold feasibility


,maximum_unigram_jaccard,eligible_records,eligible_percent,lowest_language_eligibility_percent,median_qualifying_donors,p10_qualifying_donors
0,0.00,4829,64.39,47.04,32.0,1.0
1,0.05,6295,83.93,73.92,22.0,2.0
2,0.10,7103,94.71,91.68,29.0,3.0
3,0.15,7270,96.93,94.88,37.0,5.0
4,0.20,7275,97.00,94.96,41.0,5.0
5,0.25,7278,97.04,95.04,41.0,5.0
6,0.30,7278,97.04,95.04,41.0,5.0



Lexical-threshold feasibility by language


,language,maximum_unigram_jaccard,records,eligible_records,eligible_percent,median_qualifying_donors,p10_qualifying_donors
0,go,0.00,1250,960,76.80,85.5,3.0
1,go,0.05,1250,1133,90.64,92.0,6.0
2,go,0.10,1250,1201,96.08,120.0,10.0
3,go,0.15,1250,1215,97.20,131.0,12.0
4,go,0.20,1250,1215,97.20,139.0,12.0
5,go,0.25,1250,1215,97.20,141.0,12.0
6,go,0.30,1250,1215,97.20,141.0,12.0
7,java,0.00,1250,588,47.04,15.0,1.0
8,java,0.05,1250,924,73.92,11.0,1.0
9,java,0.10,1250,1181,94.48,18.0,2.0



Saved exploratory outputs
Record audit: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\topic_swap_lexical_overlap_audit.csv
Language summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\topic_swap_lexical_overlap_by_language.csv
Overall summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\topic_swap_lexical_overlap_overall.csv


In [6]:
import csv
import gzip
import hashlib
import sqlite3
from collections import defaultdict
from datetime import datetime, timezone


CANDIDATE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed_rework_v2"
    / "01_candidate_population_index.sqlite"
)

CALIBRATION_SAMPLE_SIZE_PER_LANGUAGE = 1250
CALIBRATION_TOTAL = (
    CALIBRATION_SAMPLE_SIZE_PER_LANGUAGE
    * len(EXPECTED_LANGUAGES)
)
CALIBRATION_SEED = 20260805

CALIBRATION_SAMPLE_PATH = (
    OUTPUT_DIR
    / "02_Train_Perturbation_Calibration_Sample_7500.csv"
)

CALIBRATION_METADATA_PATH = (
    OUTPUT_DIR
    / "02_Train_Perturbation_Calibration_Sample_7500_metadata.json"
)


def stable_uint64(*parts):
    joined = "\x1f".join(
        str(part)
        for part in parts
    )

    return int.from_bytes(
        hashlib.sha256(
            joined.encode("utf-8")
        ).digest()[:8],
        byteorder="big",
        signed=False,
    )


assert CANDIDATE_DB_PATH.exists(), CANDIDATE_DB_PATH

connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    train_repositories = pd.read_sql_query(
        """
        SELECT DISTINCT
            language,
            repository
        FROM eligible_candidate_records
        WHERE original_split = 'train'
        """,
        connection,
    )

    train_repositories[
        "repository_hash"
    ] = train_repositories.apply(
        lambda row: stable_uint64(
            CALIBRATION_SEED,
            row["language"],
            row["repository"],
        ),
        axis=1,
    )

    selected_repositories = (
        train_repositories
        .sort_values(
            [
                "language",
                "repository_hash",
                "repository",
            ]
        )
        .groupby(
            "language",
            group_keys=False,
        )
        .head(
            CALIBRATION_SAMPLE_SIZE_PER_LANGUAGE
        )
        .reset_index(drop=True)
    )

    selected_repository_counts = (
        selected_repositories[
            "language"
        ]
        .value_counts()
        .reindex(EXPECTED_LANGUAGES)
    )

    assert (
        selected_repository_counts
        == CALIBRATION_SAMPLE_SIZE_PER_LANGUAGE
    ).all()

    connection.execute(
        """
        CREATE TEMP TABLE
        selected_calibration_repositories (
            language TEXT NOT NULL,
            repository TEXT NOT NULL,
            PRIMARY KEY (
                language,
                repository
            )
        )
        """
    )

    connection.executemany(
        """
        INSERT INTO selected_calibration_repositories (
            language,
            repository
        )
        VALUES (?, ?)
        """,
        selected_repositories[
            [
                "language",
                "repository",
            ]
        ].itertuples(
            index=False,
            name=None,
        ),
    )

    repository_candidates = pd.read_sql_query(
        """
        SELECT
            e.family_id,
            e.language,
            e.repository,
            e.original_split,
            e.file_path,
            e.function_name,
            e.commit_sha,
            e.source_url,
            e.shard_relative_path,
            e.shard_line_number,
            e.code_token_count,
            e.comment_token_count,
            e.combined_token_count,
            e.code_line_count,
            e.comment_line_count,
            e.exact_code_hash,
            e.exact_comment_hash,
            e.exact_pair_hash,
            e.pair_token_count_capped
        FROM eligible_candidate_records AS e
        INNER JOIN selected_calibration_repositories AS s
            ON e.language = s.language
           AND e.repository = s.repository
        WHERE e.original_split = 'train'
        """,
        connection,
    )

finally:
    connection.close()


repository_candidates[
    "method_hash"
] = repository_candidates[
    "family_id"
].map(
    lambda family_id: stable_uint64(
        CALIBRATION_SEED,
        family_id,
    )
)

calibration_sample = (
    repository_candidates
    .sort_values(
        [
            "language",
            "repository",
            "method_hash",
            "family_id",
        ]
    )
    .groupby(
        [
            "language",
            "repository",
        ],
        group_keys=False,
    )
    .head(1)
    .copy()
)

calibration_sample[
    "repository_hash"
] = calibration_sample.apply(
    lambda row: stable_uint64(
        CALIBRATION_SEED,
        row["language"],
        row["repository"],
    ),
    axis=1,
)

calibration_sample = (
    calibration_sample
    .sort_values(
        [
            "language",
            "repository_hash",
            "method_hash",
        ]
    )
    .reset_index(drop=True)
)

calibration_sample[
    "calibration_order"
] = (
    calibration_sample
    .groupby("language")
    .cumcount()
    + 1
)

calibration_sample[
    "calibration_id"
] = calibration_sample.apply(
    lambda row: (
        f"CAL-{row['language'].upper()}-"
        f"{int(row['calibration_order']):04d}"
    ),
    axis=1,
)


targets_by_shard = defaultdict(dict)

for row in calibration_sample.itertuples(
    index=False
):
    targets_by_shard[
        row.shard_relative_path
    ][int(row.shard_line_number)] = (
        row.family_id
    )


expected_hashes = {
    row.family_id: (
        row.exact_code_hash,
        row.exact_comment_hash,
    )
    for row in calibration_sample.itertuples(
        index=False
    )
}

raw_text_by_family = {}

for shard_relative_path, line_targets in (
    targets_by_shard.items()
):
    shard_path = (
        PROJECT_ROOT
        / shard_relative_path
    )

    found_records = 0

    with gzip.open(
        shard_path,
        mode="rt",
        encoding="utf-8",
        errors="strict",
    ) as handle:
        for line_number, raw_line in enumerate(
            handle,
            start=1,
        ):
            if line_number not in line_targets:
                continue

            record = json.loads(raw_line)
            family_id = line_targets[line_number]

            code = record["code"]
            comment = record["docstring"]

            code_hash = hashlib.sha256(
                code.encode("utf-8")
            ).hexdigest()

            comment_hash = hashlib.sha256(
                comment.encode("utf-8")
            ).hexdigest()

            assert (
                code_hash,
                comment_hash,
            ) == expected_hashes[family_id]

            raw_text_by_family[family_id] = {
                "original_code": code,
                "original_comment": comment,
            }

            found_records += 1

            if found_records == len(
                line_targets
            ):
                break

    assert found_records == len(
        line_targets
    )


calibration_sample[
    "original_code"
] = calibration_sample[
    "family_id"
].map(
    lambda family_id: raw_text_by_family[
        family_id
    ]["original_code"]
)

calibration_sample[
    "original_comment"
] = calibration_sample[
    "family_id"
].map(
    lambda family_id: raw_text_by_family[
        family_id
    ]["original_comment"]
)

calibration_sample[
    "derived_physical_comment_lines"
] = calibration_sample[
    "original_comment"
].map(
    count_physical_splitlines
)

assert (
    calibration_sample[
        "comment_line_count"
    ]
    == calibration_sample[
        "derived_physical_comment_lines"
    ]
).all()

assert len(calibration_sample) == CALIBRATION_TOTAL
assert (
    calibration_sample[
        "repository"
    ].groupby(
        calibration_sample["language"]
    ).nunique()
    == CALIBRATION_SAMPLE_SIZE_PER_LANGUAGE
).all()

assert (
    calibration_sample[
        "original_split"
    ] == "train"
).all()

assert calibration_sample[
    "pair_token_count_capped"
].le(512).all()

assert not calibration_sample[
    "family_id"
].duplicated().any()

assert not calibration_sample[
    "repository"
].groupby(
    calibration_sample["language"]
).apply(
    lambda values: values.duplicated().any()
).any()


export_columns = [
    "calibration_id",
    "family_id",
    "language",
    "calibration_order",
    "repository",
    "original_split",
    "file_path",
    "function_name",
    "commit_sha",
    "source_url",
    "shard_relative_path",
    "shard_line_number",
    "code_token_count",
    "comment_token_count",
    "combined_token_count",
    "pair_token_count_capped",
    "code_line_count",
    "comment_line_count",
    "exact_code_hash",
    "exact_comment_hash",
    "exact_pair_hash",
    "original_code",
    "original_comment",
]

calibration_export = (
    calibration_sample[
        export_columns
    ]
    .sort_values(
        [
            "language",
            "calibration_order",
        ]
    )
    .reset_index(drop=True)
)

calibration_export.to_csv(
    CALIBRATION_SAMPLE_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)


calibration_summary = (
    calibration_export
    .groupby(
        "language",
        as_index=False,
    )
    .agg(
        records=(
            "family_id",
            "count",
        ),
        repositories=(
            "repository",
            "nunique",
        ),
        minimum_comment_lines=(
            "comment_line_count",
            "min",
        ),
        median_comment_lines=(
            "comment_line_count",
            "median",
        ),
        maximum_comment_lines=(
            "comment_line_count",
            "max",
        ),
        maximum_codebert_pair_tokens=(
            "pair_token_count_capped",
            "max",
        ),
    )
)


calibration_metadata = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "partition": "train",
    "purpose": (
        "Perturbation-rule calibration without "
        "using validation or test outcomes"
    ),
    "selection_seed": CALIBRATION_SEED,
    "sample_size_per_language": (
        CALIBRATION_SAMPLE_SIZE_PER_LANGUAGE
    ),
    "total_sample_size": CALIBRATION_TOTAL,
    "repository_rule": (
        "one deterministically selected method "
        "from each selected repository"
    ),
    "source_database": str(
        CANDIDATE_DB_PATH
    ),
    "output_csv": str(
        CALIBRATION_SAMPLE_PATH
    ),
}

with CALIBRATION_METADATA_PATH.open(
    mode="w",
    encoding="utf-8",
) as handle:
    json.dump(
        calibration_metadata,
        handle,
        indent=2,
        ensure_ascii=False,
    )


print("Training calibration sample")
display(calibration_summary)

print(
    "\nCalibration CSV:",
    CALIBRATION_SAMPLE_PATH,
)

print(
    "Metadata JSON:",
    CALIBRATION_METADATA_PATH,
)

print(
    "CSV size:",
    (
        f"{CALIBRATION_SAMPLE_PATH.stat().st_size / 1_000_000:.2f} MB"
    ),
)

print(
    "\nTRAINING CALIBRATION SAMPLE CREATED"
)

Training calibration sample


,language,records,repositories,minimum_comment_lines,median_comment_lines,maximum_comment_lines,maximum_codebert_pair_tokens
0,go,1250,1250,1,1.0,34,510
1,java,1250,1250,1,4.0,32,512
2,javascript,1250,1250,1,1.0,41,511
3,php,1250,1250,1,4.0,45,511
4,python,1250,1250,1,2.0,21,512
5,ruby,1250,1250,1,2.0,40,509



Calibration CSV: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_Train_Perturbation_Calibration_Sample_7500.csv
Metadata JSON: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_Train_Perturbation_Calibration_Sample_7500_metadata.json
CSV size: 9.12 MB

TRAINING CALIBRATION SAMPLE CREATED


In [7]:
import json
import math
import re
from bisect import bisect_right
from collections import defaultdict
from datetime import datetime, timezone

import pandas as pd


TOPIC_CALIBRATION_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "rework_v2"
    / "02_distractor_generation"
    / "train_calibration"
)

TOPIC_CALIBRATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

TOPIC_TARGET_AUDIT_PATH = (
    TOPIC_CALIBRATION_DIR
    / "topic_swap_combination_target_audit.csv"
)

TOPIC_LANGUAGE_SUMMARY_PATH = (
    TOPIC_CALIBRATION_DIR
    / "topic_swap_combination_by_language.csv"
)

TOPIC_OVERALL_SUMMARY_PATH = (
    TOPIC_CALIBRATION_DIR
    / "topic_swap_combination_overall.csv"
)

TOPIC_DECISION_PATH = (
    TOPIC_CALIBRATION_DIR
    / "topic_swap_rule_decision.json"
)

MINIMUM_LANGUAGE_ELIGIBILITY_PERCENT = 95.0

LEXICAL_THRESHOLDS = (
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
)


def physical_line_count(text):
    return max(
        len(str(text).splitlines()),
        1,
    )


def surface_tokens(text):
    return re.findall(
        r"\S+",
        str(text),
        flags=re.UNICODE,
    )


def normalized_lexical_tokens(text):
    return frozenset(
        token.casefold()
        for token in re.findall(
            r"\w+",
            str(text),
            flags=re.UNICODE,
        )
        if token.strip("_")
    )


def unigram_jaccard(left_tokens, right_tokens):
    union = left_tokens | right_tokens

    if not union:
        return 1.0

    return (
        len(left_tokens & right_tokens)
        / len(union)
    )


def exact_token_caliper(token_count):
    return token_count, token_count


def absolute_one_caliper(token_count):
    return (
        max(0, token_count - 1),
        token_count + 1,
    )


def absolute_two_caliper(token_count):
    return (
        max(0, token_count - 2),
        token_count + 2,
    )


def hybrid_ten_percent_caliper(token_count):
    tolerance = max(
        2,
        math.ceil(token_count * 0.10),
    )

    return (
        max(0, token_count - tolerance),
        token_count + tolerance,
    )


def hybrid_fifteen_percent_caliper(token_count):
    tolerance = max(
        3,
        math.ceil(token_count * 0.15),
    )

    return (
        max(0, token_count - tolerance),
        token_count + tolerance,
    )


CALIPER_RULES = {
    "exact_token_count": {
        "order": 0,
        "function": exact_token_caliper,
    },
    "absolute_plus_minus_1": {
        "order": 1,
        "function": absolute_one_caliper,
    },
    "absolute_plus_minus_2": {
        "order": 2,
        "function": absolute_two_caliper,
    },
    "hybrid_10_percent_minimum_2": {
        "order": 3,
        "function": hybrid_ten_percent_caliper,
    },
    "hybrid_15_percent_minimum_3": {
        "order": 4,
        "function": hybrid_fifteen_percent_caliper,
    },
}


calibration = pd.read_csv(
    CALIBRATION_SAMPLE_PATH,
    keep_default_na=False,
)

assert len(calibration) == 7500
assert (
    calibration["original_split"] == "train"
).all()
assert calibration["family_id"].is_unique

calibration[
    "physical_line_count"
] = calibration[
    "original_comment"
].map(
    physical_line_count
)

calibration[
    "surface_token_count"
] = calibration[
    "original_comment"
].map(
    lambda text: len(surface_tokens(text))
)

calibration[
    "lexical_token_set"
] = calibration[
    "original_comment"
].map(
    normalized_lexical_tokens
)

assert (
    calibration["physical_line_count"]
    == calibration["comment_line_count"]
).all()


candidate_groups = defaultdict(list)

for row in calibration.itertuples(
    index=False,
):
    group_key = (
        row.language,
        int(row.physical_line_count),
    )

    candidate_groups[group_key].append(
        {
            "family_id": row.family_id,
            "repository": row.repository,
            "exact_comment_hash": (
                row.exact_comment_hash
            ),
            "surface_token_count": int(
                row.surface_token_count
            ),
            "lexical_token_set": (
                row.lexical_token_set
            ),
        }
    )


target_rows = []

for target in calibration.itertuples(
    index=False,
):
    group_key = (
        target.language,
        int(target.physical_line_count),
    )

    possible_donors = []

    for donor in candidate_groups[group_key]:
        if (
            donor["repository"]
            == target.repository
        ):
            continue

        if (
            donor["exact_comment_hash"]
            == target.exact_comment_hash
        ):
            continue

        lexical_overlap = unigram_jaccard(
            target.lexical_token_set,
            donor["lexical_token_set"],
        )

        possible_donors.append(
            (
                donor["surface_token_count"],
                lexical_overlap,
            )
        )

    target_result = {
        "calibration_id": (
            target.calibration_id
        ),
        "family_id": target.family_id,
        "language": target.language,
        "repository": target.repository,
        "physical_line_count": int(
            target.physical_line_count
        ),
        "surface_token_count": int(
            target.surface_token_count
        ),
        "different_repository_donors": len(
            possible_donors
        ),
    }

    for rule_name, rule_details in (
        CALIPER_RULES.items()
    ):
        lower_bound, upper_bound = (
            rule_details["function"](
                int(target.surface_token_count)
            )
        )

        eligible_jaccards = sorted(
            lexical_overlap
            for (
                donor_token_count,
                lexical_overlap,
            ) in possible_donors
            if (
                lower_bound
                <= donor_token_count
                <= upper_bound
            )
        )

        for threshold in LEXICAL_THRESHOLDS:
            threshold_label = (
                f"{threshold:.2f}"
                .replace(".", "_")
            )

            count_column = (
                f"{rule_name}"
                f"__jaccard_{threshold_label}"
                f"__candidate_count"
            )

            target_result[count_column] = (
                bisect_right(
                    eligible_jaccards,
                    threshold,
                )
            )

    target_rows.append(target_result)


topic_target_audit = pd.DataFrame(
    target_rows
)


language_rows = []

for rule_name, rule_details in (
    CALIPER_RULES.items()
):
    for threshold in LEXICAL_THRESHOLDS:
        threshold_label = (
            f"{threshold:.2f}"
            .replace(".", "_")
        )

        count_column = (
            f"{rule_name}"
            f"__jaccard_{threshold_label}"
            f"__candidate_count"
        )

        for language_value, language_frame in (
            topic_target_audit.groupby(
                "language",
                sort=True,
            )
        ):
            donor_counts = language_frame[
                count_column
            ]

            language_rows.append(
                {
                    "language": language_value,
                    "caliper_rule": rule_name,
                    "caliper_order": (
                        rule_details["order"]
                    ),
                    "maximum_unigram_jaccard": (
                        threshold
                    ),
                    "records": len(
                        language_frame
                    ),
                    "eligible_records": int(
                        donor_counts.gt(0).sum()
                    ),
                    "eligible_percent": round(
                        100
                        * donor_counts.gt(0).mean(),
                        2,
                    ),
                    "median_candidate_count": float(
                        donor_counts.median()
                    ),
                    "p10_candidate_count": float(
                        donor_counts.quantile(
                            0.10
                        )
                    ),
                    "minimum_candidate_count": int(
                        donor_counts.min()
                    ),
                }
            )


topic_language_summary = pd.DataFrame(
    language_rows
)


overall_rows = []

for rule_name, rule_details in (
    CALIPER_RULES.items()
):
    for threshold in LEXICAL_THRESHOLDS:
        threshold_label = (
            f"{threshold:.2f}"
            .replace(".", "_")
        )

        count_column = (
            f"{rule_name}"
            f"__jaccard_{threshold_label}"
            f"__candidate_count"
        )

        donor_counts = topic_target_audit[
            count_column
        ]

        combination_languages = (
            topic_language_summary.loc[
                (
                    topic_language_summary[
                        "caliper_rule"
                    ] == rule_name
                )
                & (
                    topic_language_summary[
                        "maximum_unigram_jaccard"
                    ] == threshold
                )
            ]
        )

        lowest_language_percent = float(
            combination_languages[
                "eligible_percent"
            ].min()
        )

        overall_rows.append(
            {
                "caliper_rule": rule_name,
                "caliper_order": (
                    rule_details["order"]
                ),
                "maximum_unigram_jaccard": (
                    threshold
                ),
                "eligible_records": int(
                    donor_counts.gt(0).sum()
                ),
                "eligible_percent": round(
                    100
                    * donor_counts.gt(0).mean(),
                    2,
                ),
                "lowest_language_eligibility_percent": (
                    lowest_language_percent
                ),
                "median_candidate_count": float(
                    donor_counts.median()
                ),
                "p10_candidate_count": float(
                    donor_counts.quantile(
                        0.10
                    )
                ),
                "passes_language_requirement": (
                    lowest_language_percent
                    >= MINIMUM_LANGUAGE_ELIGIBILITY_PERCENT
                ),
            }
        )


topic_overall_summary = (
    pd.DataFrame(overall_rows)
    .sort_values(
        [
            "caliper_order",
            "maximum_unigram_jaccard",
        ]
    )
    .reset_index(drop=True)
)


passing_combinations = (
    topic_overall_summary.loc[
        topic_overall_summary[
            "passes_language_requirement"
        ]
    ]
    .sort_values(
        [
            "caliper_order",
            "maximum_unigram_jaccard",
        ]
    )
)

assert not passing_combinations.empty, (
    "No topic-swap rule reaches 95% "
    "eligibility in every language."
)

selected_topic_rule = (
    passing_combinations.iloc[0]
)

SELECTED_TOPIC_CALIPER_RULE = str(
    selected_topic_rule[
        "caliper_rule"
    ]
)

SELECTED_MAXIMUM_UNIGRAM_JACCARD = float(
    selected_topic_rule[
        "maximum_unigram_jaccard"
    ]
)


selected_language_results = (
    topic_language_summary.loc[
        (
            topic_language_summary[
                "caliper_rule"
            ] == SELECTED_TOPIC_CALIPER_RULE
        )
        & (
            topic_language_summary[
                "maximum_unigram_jaccard"
            ]
            == SELECTED_MAXIMUM_UNIGRAM_JACCARD
        )
    ]
    .sort_values("language")
    .reset_index(drop=True)
)


topic_target_audit.to_csv(
    TOPIC_TARGET_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

topic_language_summary.to_csv(
    TOPIC_LANGUAGE_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)

topic_overall_summary.to_csv(
    TOPIC_OVERALL_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)


topic_rule_decision = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "calibration_partition": "train",
    "calibration_sample": str(
        CALIBRATION_SAMPLE_PATH
    ),
    "selection_rule": (
        "Choose the narrowest surface-token "
        "caliper, then the lowest unigram "
        "Jaccard threshold, that provides at "
        "least 95 percent eligibility in every "
        "programming language."
    ),
    "minimum_language_eligibility_percent": (
        MINIMUM_LANGUAGE_ELIGIBILITY_PERCENT
    ),
    "exact_physical_line_matching": True,
    "different_repository_required": True,
    "identical_comment_excluded": True,
    "selected_caliper_rule": (
        SELECTED_TOPIC_CALIPER_RULE
    ),
    "selected_maximum_unigram_jaccard": (
        SELECTED_MAXIMUM_UNIGRAM_JACCARD
    ),
    "selected_overall_eligibility_percent": float(
        selected_topic_rule[
            "eligible_percent"
        ]
    ),
    "selected_lowest_language_eligibility_percent": float(
        selected_topic_rule[
            "lowest_language_eligibility_percent"
        ]
    ),
    "output_files": {
        "target_audit": str(
            TOPIC_TARGET_AUDIT_PATH
        ),
        "language_summary": str(
            TOPIC_LANGUAGE_SUMMARY_PATH
        ),
        "overall_summary": str(
            TOPIC_OVERALL_SUMMARY_PATH
        ),
    },
}

with TOPIC_DECISION_PATH.open(
    mode="w",
    encoding="utf-8",
) as handle:
    json.dump(
        topic_rule_decision,
        handle,
        indent=2,
        ensure_ascii=False,
    )


print("Passing topic-swap combinations")
display(
    passing_combinations[
        [
            "caliper_rule",
            "maximum_unigram_jaccard",
            "eligible_records",
            "eligible_percent",
            "lowest_language_eligibility_percent",
            "median_candidate_count",
            "p10_candidate_count",
        ]
    ].reset_index(drop=True)
)

print("\nSelected topic-swap rule")
print(
    "Surface-token caliper:",
    SELECTED_TOPIC_CALIPER_RULE,
)
print(
    "Maximum unigram Jaccard:",
    SELECTED_MAXIMUM_UNIGRAM_JACCARD,
)

print("\nSelected rule by language")
display(
    selected_language_results[
        [
            "language",
            "records",
            "eligible_records",
            "eligible_percent",
            "median_candidate_count",
            "p10_candidate_count",
        ]
    ]
)

print("\nSaved calibration outputs")
print("Target audit:", TOPIC_TARGET_AUDIT_PATH)
print(
    "Language summary:",
    TOPIC_LANGUAGE_SUMMARY_PATH,
)
print(
    "Overall summary:",
    TOPIC_OVERALL_SUMMARY_PATH,
)
print("Rule decision:", TOPIC_DECISION_PATH)

Passing topic-swap combinations


,caliper_rule,maximum_unigram_jaccard,eligible_records,eligible_percent,lowest_language_eligibility_percent,median_candidate_count,p10_candidate_count
0,hybrid_10_percent_minimum_2,0.15,7249,96.65,95.68,34.0,3.0
1,hybrid_10_percent_minimum_2,0.20,7270,96.93,96.00,38.0,4.0
2,hybrid_10_percent_minimum_2,0.25,7273,96.97,96.00,39.0,4.0
3,hybrid_10_percent_minimum_2,0.30,7274,96.99,96.00,39.0,4.0
4,hybrid_15_percent_minimum_3,0.15,7308,97.44,96.96,47.0,5.0
5,hybrid_15_percent_minimum_3,0.20,7324,97.65,97.12,52.0,6.0
6,hybrid_15_percent_minimum_3,0.25,7326,97.68,97.12,53.0,6.0
7,hybrid_15_percent_minimum_3,0.30,7327,97.69,97.12,53.0,6.0



Selected topic-swap rule
Surface-token caliper: hybrid_10_percent_minimum_2
Maximum unigram Jaccard: 0.15

Selected rule by language


,language,records,eligible_records,eligible_percent,median_candidate_count,p10_candidate_count
0,go,1250,1212,96.96,114.0,5.0
1,java,1250,1205,96.40,24.0,3.0
2,javascript,1250,1205,96.40,23.0,3.0
3,php,1250,1215,97.20,31.0,4.0
4,python,1250,1216,97.28,41.0,5.0
5,ruby,1250,1196,95.68,27.0,2.0



Saved calibration outputs
Target audit: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\train_calibration\topic_swap_combination_target_audit.csv
Language summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\train_calibration\topic_swap_combination_by_language.csv
Overall summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\train_calibration\topic_swap_combination_overall.csv
Rule decision: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\train_calibration\topic_swap_rule_decision.json


In [8]:
import hashlib
import json
import random
import re
from collections import Counter
from datetime import datetime, timezone

import pandas as pd


SHUFFLE_CALIBRATION_DIR = (
    PROJECT_ROOT
    / "outputs"
    / "rework_v2"
    / "02_distractor_generation"
    / "train_calibration"
)

SHUFFLE_CALIBRATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

SHUFFLE_SEED = 20260805
RANDOM_SHUFFLE_ATTEMPTS = 64
MINIMUM_SHUFFLEABLE_COVERAGE_PERCENT = 95.0

BIGRAM_THRESHOLDS = (
    0.00,
    0.05,
    0.10,
    0.15,
    0.20,
    0.25,
    0.30,
    0.40,
    0.50,
)

SHUFFLE_AUDIT_PATH = (
    SHUFFLE_CALIBRATION_DIR
    / "grammar_shuffle_candidate_audit.csv"
)

SHUFFLE_LANGUAGE_SUMMARY_PATH = (
    SHUFFLE_CALIBRATION_DIR
    / "grammar_shuffle_threshold_by_language.csv"
)

SHUFFLE_OVERALL_SUMMARY_PATH = (
    SHUFFLE_CALIBRATION_DIR
    / "grammar_shuffle_threshold_overall.csv"
)

SHUFFLE_DECISION_PATH = (
    SHUFFLE_CALIBRATION_DIR
    / "grammar_shuffle_rule_decision.json"
)


def split_token_layout(text):
    chunks = re.split(
        r"(\s+)",
        str(text),
        flags=re.UNICODE,
    )

    token_positions = [
        index
        for index, chunk in enumerate(chunks)
        if chunk and not chunk.isspace()
    ]

    tokens = [
        chunks[index]
        for index in token_positions
    ]

    return chunks, token_positions, tokens


def render_token_permutation(
    chunks,
    token_positions,
    permuted_tokens,
):
    rendered_chunks = list(chunks)

    for position, token in zip(
        token_positions,
        permuted_tokens,
    ):
        rendered_chunks[position] = token

    return "".join(rendered_chunks)


def normalized_token(token):
    return str(token).casefold()


def multiset_bigram_jaccard(
    original_tokens,
    candidate_tokens,
):
    original_normalized = [
        normalized_token(token)
        for token in original_tokens
    ]

    candidate_normalized = [
        normalized_token(token)
        for token in candidate_tokens
    ]

    if len(original_normalized) < 2:
        return 1.0

    original_bigrams = Counter(
        zip(
            original_normalized[:-1],
            original_normalized[1:],
        )
    )

    candidate_bigrams = Counter(
        zip(
            candidate_normalized[:-1],
            candidate_normalized[1:],
        )
    )

    all_bigrams = (
        set(original_bigrams)
        | set(candidate_bigrams)
    )

    intersection = sum(
        min(
            original_bigrams[bigram],
            candidate_bigrams[bigram],
        )
        for bigram in all_bigrams
    )

    union = sum(
        max(
            original_bigrams[bigram],
            candidate_bigrams[bigram],
        )
        for bigram in all_bigrams
    )

    if union == 0:
        return 1.0

    return intersection / union


def fixed_position_fraction(
    original_tokens,
    candidate_tokens,
):
    if not original_tokens:
        return 1.0

    return sum(
        normalized_token(original)
        == normalized_token(candidate)
        for original, candidate in zip(
            original_tokens,
            candidate_tokens,
        )
    ) / len(original_tokens)


def deterministic_seed(
    family_id,
    attempt,
):
    value = (
        f"{SHUFFLE_SEED}\x1f"
        f"{family_id}\x1f"
        f"{attempt}"
    )

    return int.from_bytes(
        hashlib.sha256(
            value.encode("utf-8")
        ).digest()[:8],
        byteorder="big",
        signed=False,
    )


def generate_candidate_permutations(
    tokens,
    family_id,
):
    original = tuple(tokens)
    token_count = len(original)

    candidates = set()

    if token_count < 2:
        return candidates

    reversed_tokens = tuple(
        reversed(original)
    )

    if reversed_tokens != original:
        candidates.add(reversed_tokens)

    maximum_rotations = min(
        token_count - 1,
        12,
    )

    for shift in range(
        1,
        maximum_rotations + 1,
    ):
        rotated = (
            original[shift:]
            + original[:shift]
        )

        if rotated != original:
            candidates.add(rotated)

    even_odd = (
        original[::2]
        + original[1::2]
    )

    odd_even = (
        original[1::2]
        + original[::2]
    )

    if even_odd != original:
        candidates.add(even_odd)

    if odd_even != original:
        candidates.add(odd_even)

    for attempt in range(
        RANDOM_SHUFFLE_ATTEMPTS
    ):
        shuffled = list(original)

        generator = random.Random(
            deterministic_seed(
                family_id,
                attempt,
            )
        )

        generator.shuffle(shuffled)
        shuffled = tuple(shuffled)

        if shuffled != original:
            candidates.add(shuffled)

    return candidates


calibration = pd.read_csv(
    CALIBRATION_SAMPLE_PATH,
    keep_default_na=False,
)

assert len(calibration) == 7500
assert calibration["family_id"].is_unique
assert (
    calibration["original_split"] == "train"
).all()


audit_rows = []

for row in calibration.itertuples(
    index=False,
):
    chunks, token_positions, tokens = (
        split_token_layout(
            row.original_comment
        )
    )

    candidate_permutations = (
        generate_candidate_permutations(
            tokens,
            row.family_id,
        )
    )

    evaluated_candidates = []

    for candidate_tokens in (
        candidate_permutations
    ):
        shuffled_comment = (
            render_token_permutation(
                chunks,
                token_positions,
                candidate_tokens,
            )
        )

        if shuffled_comment == row.original_comment:
            continue

        bigram_jaccard = (
            multiset_bigram_jaccard(
                tokens,
                candidate_tokens,
            )
        )

        position_fraction = (
            fixed_position_fraction(
                tokens,
                candidate_tokens,
            )
        )

        evaluated_candidates.append(
            {
                "tokens": candidate_tokens,
                "comment": shuffled_comment,
                "bigram_jaccard": (
                    bigram_jaccard
                ),
                "fixed_position_fraction": (
                    position_fraction
                ),
            }
        )

    evaluated_candidates.sort(
        key=lambda candidate: (
            candidate["bigram_jaccard"],
            candidate[
                "fixed_position_fraction"
            ],
            candidate["tokens"],
        )
    )

    structurally_shuffleable = bool(
        evaluated_candidates
    )

    if structurally_shuffleable:
        best_candidate = (
            evaluated_candidates[0]
        )

        best_comment = (
            best_candidate["comment"]
        )

        best_bigram_jaccard = float(
            best_candidate[
                "bigram_jaccard"
            ]
        )

        best_fixed_position_fraction = (
            float(
                best_candidate[
                    "fixed_position_fraction"
                ]
            )
        )
    else:
        best_comment = ""
        best_bigram_jaccard = None
        best_fixed_position_fraction = None

    original_physical_lines = max(
        len(
            str(
                row.original_comment
            ).splitlines()
        ),
        1,
    )

    shuffled_physical_lines = (
        max(
            len(best_comment.splitlines()),
            1,
        )
        if structurally_shuffleable
        else None
    )

    shuffled_chunks, _, shuffled_tokens = (
        split_token_layout(best_comment)
        if structurally_shuffleable
        else (
            [],
            [],
            [],
        )
    )

    audit_result = {
        "calibration_id": (
            row.calibration_id
        ),
        "family_id": row.family_id,
        "language": row.language,
        "repository": row.repository,
        "surface_token_count": len(
            tokens
        ),
        "unique_casefolded_token_count": len(
            {
                normalized_token(token)
                for token in tokens
            }
        ),
        "candidate_permutation_count": len(
            evaluated_candidates
        ),
        "structurally_shuffleable": (
            structurally_shuffleable
        ),
        "best_bigram_jaccard": (
            best_bigram_jaccard
        ),
        "best_fixed_position_fraction": (
            best_fixed_position_fraction
        ),
        "original_physical_lines": (
            original_physical_lines
        ),
        "shuffled_physical_lines": (
            shuffled_physical_lines
        ),
        "surface_tokens_preserved": (
            Counter(tokens)
            == Counter(shuffled_tokens)
            if structurally_shuffleable
            else False
        ),
        "exact_whitespace_preserved": (
            re.findall(
                r"\s+",
                row.original_comment,
                flags=re.UNICODE,
            )
            == re.findall(
                r"\s+",
                best_comment,
                flags=re.UNICODE,
            )
            if structurally_shuffleable
            else False
        ),
        "best_shuffled_comment": (
            best_comment
        ),
    }

    for threshold in BIGRAM_THRESHOLDS:
        threshold_label = (
            f"{threshold:.2f}"
            .replace(".", "_")
        )

        audit_result[
            f"eligible_at_most_{threshold_label}"
        ] = (
            structurally_shuffleable
            and best_bigram_jaccard
            <= threshold
        )

    audit_rows.append(
        audit_result
    )


shuffle_audit = pd.DataFrame(
    audit_rows
)


assert (
    shuffle_audit.loc[
        shuffle_audit[
            "structurally_shuffleable"
        ],
        "surface_tokens_preserved",
    ]
).all()

assert (
    shuffle_audit.loc[
        shuffle_audit[
            "structurally_shuffleable"
        ],
        "exact_whitespace_preserved",
    ]
).all()

assert (
    shuffle_audit.loc[
        shuffle_audit[
            "structurally_shuffleable"
        ],
        "original_physical_lines",
    ].to_numpy()
    == shuffle_audit.loc[
        shuffle_audit[
            "structurally_shuffleable"
        ],
        "shuffled_physical_lines",
    ].to_numpy()
).all()


language_rows = []

for language_value, language_frame in (
    shuffle_audit.groupby(
        "language",
        sort=True,
    )
):
    shuffleable_records = int(
        language_frame[
            "structurally_shuffleable"
        ].sum()
    )

    for threshold in BIGRAM_THRESHOLDS:
        threshold_label = (
            f"{threshold:.2f}"
            .replace(".", "_")
        )

        eligible_column = (
            f"eligible_at_most_"
            f"{threshold_label}"
        )

        eligible_records = int(
            language_frame[
                eligible_column
            ].sum()
        )

        language_rows.append(
            {
                "language": language_value,
                "maximum_bigram_jaccard": (
                    threshold
                ),
                "records": len(
                    language_frame
                ),
                "structurally_shuffleable_records": (
                    shuffleable_records
                ),
                "structurally_shuffleable_percent": round(
                    100
                    * shuffleable_records
                    / len(language_frame),
                    2,
                ),
                "eligible_records": (
                    eligible_records
                ),
                "eligible_percent_of_all": round(
                    100
                    * eligible_records
                    / len(language_frame),
                    2,
                ),
                "eligible_percent_of_shuffleable": (
                    round(
                        100
                        * eligible_records
                        / shuffleable_records,
                        2,
                    )
                    if shuffleable_records
                    else 0.0
                ),
            }
        )


shuffle_language_summary = pd.DataFrame(
    language_rows
)


overall_rows = []

for threshold in BIGRAM_THRESHOLDS:
    threshold_label = (
        f"{threshold:.2f}"
        .replace(".", "_")
    )

    eligible_column = (
        f"eligible_at_most_{threshold_label}"
    )

    eligible_records = int(
        shuffle_audit[
            eligible_column
        ].sum()
    )

    shuffleable_records = int(
        shuffle_audit[
            "structurally_shuffleable"
        ].sum()
    )

    language_threshold = (
        shuffle_language_summary.loc[
            shuffle_language_summary[
                "maximum_bigram_jaccard"
            ] == threshold
        ]
    )

    lowest_language_coverage = float(
        language_threshold[
            "eligible_percent_of_shuffleable"
        ].min()
    )

    overall_rows.append(
        {
            "maximum_bigram_jaccard": (
                threshold
            ),
            "records": len(
                shuffle_audit
            ),
            "structurally_shuffleable_records": (
                shuffleable_records
            ),
            "eligible_records": (
                eligible_records
            ),
            "eligible_percent_of_all": round(
                100
                * eligible_records
                / len(shuffle_audit),
                2,
            ),
            "eligible_percent_of_shuffleable": round(
                100
                * eligible_records
                / shuffleable_records,
                2,
            ),
            "lowest_language_shuffleable_coverage_percent": (
                lowest_language_coverage
            ),
            "passes_language_requirement": (
                lowest_language_coverage
                >= MINIMUM_SHUFFLEABLE_COVERAGE_PERCENT
            ),
        }
    )


shuffle_overall_summary = pd.DataFrame(
    overall_rows
)


passing_thresholds = (
    shuffle_overall_summary.loc[
        shuffle_overall_summary[
            "passes_language_requirement"
        ]
    ]
    .sort_values(
        "maximum_bigram_jaccard"
    )
)

assert not passing_thresholds.empty, (
    "No bigram threshold provides at least "
    "95% coverage of shuffleable comments "
    "in every language."
)

selected_shuffle_rule = (
    passing_thresholds.iloc[0]
)

SELECTED_MAXIMUM_BIGRAM_JACCARD = float(
    selected_shuffle_rule[
        "maximum_bigram_jaccard"
    ]
)


selected_language_results = (
    shuffle_language_summary.loc[
        shuffle_language_summary[
            "maximum_bigram_jaccard"
        ]
        == SELECTED_MAXIMUM_BIGRAM_JACCARD
    ]
    .sort_values("language")
    .reset_index(drop=True)
)


shuffle_audit.to_csv(
    SHUFFLE_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

shuffle_language_summary.to_csv(
    SHUFFLE_LANGUAGE_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)

shuffle_overall_summary.to_csv(
    SHUFFLE_OVERALL_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)


shuffle_rule_decision = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "calibration_partition": "train",
    "calibration_sample": str(
        CALIBRATION_SAMPLE_PATH
    ),
    "selection_rule": (
        "Choose the lowest maximum multiset "
        "bigram-Jaccard threshold providing "
        "at least 95 percent coverage of "
        "structurally shuffleable comments "
        "in every programming language."
    ),
    "shuffle_seed": SHUFFLE_SEED,
    "random_attempts_per_record": (
        RANDOM_SHUFFLE_ATTEMPTS
    ),
    "minimum_language_coverage_percent": (
        MINIMUM_SHUFFLEABLE_COVERAGE_PERCENT
    ),
    "selected_maximum_bigram_jaccard": (
        SELECTED_MAXIMUM_BIGRAM_JACCARD
    ),
    "tokens_preserved_exactly": True,
    "whitespace_preserved_exactly": True,
    "physical_lines_preserved_exactly": True,
    "selected_eligibility_percent_of_all": float(
        selected_shuffle_rule[
            "eligible_percent_of_all"
        ]
    ),
    "selected_eligibility_percent_of_shuffleable": float(
        selected_shuffle_rule[
            "eligible_percent_of_shuffleable"
        ]
    ),
    "selected_lowest_language_coverage_percent": float(
        selected_shuffle_rule[
            "lowest_language_shuffleable_coverage_percent"
        ]
    ),
    "output_files": {
        "candidate_audit": str(
            SHUFFLE_AUDIT_PATH
        ),
        "language_summary": str(
            SHUFFLE_LANGUAGE_SUMMARY_PATH
        ),
        "overall_summary": str(
            SHUFFLE_OVERALL_SUMMARY_PATH
        ),
    },
}

with SHUFFLE_DECISION_PATH.open(
    mode="w",
    encoding="utf-8",
) as handle:
    json.dump(
        shuffle_rule_decision,
        handle,
        indent=2,
        ensure_ascii=False,
    )


print("Grammar-shuffle threshold comparison")
display(shuffle_overall_summary)

print("\nSelected grammar-shuffle rule")
print(
    "Maximum multiset bigram Jaccard:",
    SELECTED_MAXIMUM_BIGRAM_JACCARD,
)

print("\nSelected rule by language")
display(
    selected_language_results[
        [
            "language",
            "records",
            "structurally_shuffleable_records",
            "structurally_shuffleable_percent",
            "eligible_records",
            "eligible_percent_of_all",
            "eligible_percent_of_shuffleable",
        ]
    ]
)

print("\nSaved calibration outputs")
print("Candidate audit:", SHUFFLE_AUDIT_PATH)
print(
    "Language summary:",
    SHUFFLE_LANGUAGE_SUMMARY_PATH,
)
print(
    "Overall summary:",
    SHUFFLE_OVERALL_SUMMARY_PATH,
)
print("Rule decision:", SHUFFLE_DECISION_PATH)

Grammar-shuffle threshold comparison


,maximum_bigram_jaccard,records,structurally_shuffleable_records,eligible_records,eligible_percent_of_all,eligible_percent_of_shuffleable,lowest_language_shuffleable_coverage_percent,passes_language_requirement
0,0.00,7500,7235,6988,93.17,96.59,92.54,False
1,0.05,7500,7235,7223,96.31,99.83,99.50,True
2,0.10,7500,7235,7228,96.37,99.90,99.66,True
3,0.15,7500,7235,7229,96.39,99.92,99.66,True
4,0.20,7500,7235,7229,96.39,99.92,99.66,True
5,0.25,7500,7235,7229,96.39,99.92,99.66,True
6,0.30,7500,7235,7229,96.39,99.92,99.66,True
7,0.40,7500,7235,7234,96.45,99.99,99.92,True
8,0.50,7500,7235,7235,96.47,100.00,100.00,True



Selected grammar-shuffle rule
Maximum multiset bigram Jaccard: 0.05

Selected rule by language


,language,records,structurally_shuffleable_records,structurally_shuffleable_percent,eligible_records,eligible_percent_of_all,eligible_percent_of_shuffleable
0,go,1250,1248,99.84,1246,99.68,99.84
1,java,1250,1180,94.40,1178,94.24,99.83
2,javascript,1250,1192,95.36,1186,94.88,99.50
3,php,1250,1156,92.48,1155,92.40,99.91
4,python,1250,1246,99.68,1246,99.68,100.00
5,ruby,1250,1213,97.04,1212,96.96,99.92



Saved calibration outputs
Candidate audit: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\train_calibration\grammar_shuffle_candidate_audit.csv
Language summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\train_calibration\grammar_shuffle_threshold_by_language.csv
Overall summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\train_calibration\grammar_shuffle_threshold_overall.csv
Rule decision: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\train_calibration\grammar_shuffle_rule_decision.json


In [9]:
import csv
import hashlib
import json
import math
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone

import pandas as pd
from transformers import AutoTokenizer


CODEBERT_CHECKPOINT = "microsoft/codebert-base"
CODEBERT_MAX_LENGTH = 512
GENERATION_SEED = 20260805

GENERATION_OUTPUT_PATH = (
    OUTPUT_DIR
    / "02_Perturbation_Generation_Draft.csv"
)

GENERATION_AUDIT_PATH = (
    EDA_DIR
    / "perturbation_generation_audit.csv"
)

GENERATION_SUMMARY_PATH = (
    EDA_DIR
    / "perturbation_generation_by_language.csv"
)

GENERATION_MANIFEST_PATH = (
    EDA_DIR
    / "perturbation_generation_manifest.json"
)


with TOPIC_DECISION_PATH.open(
    mode="r",
    encoding="utf-8",
) as handle:
    topic_decision = json.load(handle)

with SHUFFLE_DECISION_PATH.open(
    mode="r",
    encoding="utf-8",
) as handle:
    shuffle_decision = json.load(handle)


assert (
    topic_decision["calibration_partition"]
    == "train"
)

assert (
    shuffle_decision["calibration_partition"]
    == "train"
)

assert (
    topic_decision["selected_caliper_rule"]
    == "hybrid_10_percent_minimum_2"
)

SELECTED_TOPIC_JACCARD = float(
    topic_decision[
        "selected_maximum_unigram_jaccard"
    ]
)

SELECTED_SHUFFLE_JACCARD = float(
    shuffle_decision[
        "selected_maximum_bigram_jaccard"
    ]
)

assert SELECTED_TOPIC_JACCARD == 0.15
assert SELECTED_SHUFFLE_JACCARD == 0.05


tokenizer = AutoTokenizer.from_pretrained(
    CODEBERT_CHECKPOINT,
    local_files_only=True,
    use_fast=True,
)

PAIR_SPECIAL_TOKENS = (
    tokenizer.num_special_tokens_to_add(
        pair=True
    )
)

assert PAIR_SPECIAL_TOKENS == 4


def stable_rank(*parts):
    value = "\x1f".join(
        str(part)
        for part in parts
    )

    return int.from_bytes(
        hashlib.sha256(
            value.encode("utf-8")
        ).digest()[:8],
        byteorder="big",
        signed=False,
    )


def batched_token_lengths(
    texts,
    batch_size=128,
):
    lengths = []

    text_list = [
        str(text)
        for text in texts
    ]

    for start in range(
        0,
        len(text_list),
        batch_size,
    ):
        batch = text_list[
            start:start + batch_size
        ]

        encoded = tokenizer(
            batch,
            add_special_tokens=False,
            truncation=False,
            return_length=True,
        )

        lengths.extend(
            int(length)
            for length in encoded["length"]
        )

    return lengths


generation_frame = sample.copy()

generation_frame[
    "physical_line_count"
] = generation_frame[
    "original_comment"
].map(
    physical_line_count
)

generation_frame[
    "surface_token_count"
] = generation_frame[
    "original_comment"
].map(
    lambda text: len(
        surface_tokens(text)
    )
)

generation_frame[
    "lexical_token_set"
] = generation_frame[
    "original_comment"
].map(
    normalized_lexical_tokens
)

generation_frame[
    "codebert_code_tokens"
] = batched_token_lengths(
    generation_frame[
        "original_code"
    ]
)

generation_frame[
    "codebert_comment_tokens"
] = batched_token_lengths(
    generation_frame[
        "original_comment"
    ]
)

generation_frame[
    "reconstructed_original_pair_tokens"
] = (
    generation_frame[
        "codebert_code_tokens"
    ]
    + generation_frame[
        "codebert_comment_tokens"
    ]
    + PAIR_SPECIAL_TOKENS
)

assert (
    generation_frame[
        "physical_line_count"
    ]
    == generation_frame[
        "comment_line_count"
    ]
).all()

assert (
    generation_frame[
        "reconstructed_original_pair_tokens"
    ]
    <= CODEBERT_MAX_LENGTH
).all()


record_lookup = {
    row.family_id: row
    for row in generation_frame.itertuples(
        index=False
    )
}

candidate_groups = defaultdict(list)

for row in generation_frame.itertuples(
    index=False
):
    candidate_groups[
        (
            row.language,
            int(row.physical_line_count),
        )
    ].append(
        row.family_id
    )


topic_candidate_lists = {}

for target in generation_frame.itertuples(
    index=False
):
    lower_bound, upper_bound = (
        hybrid_ten_percent_caliper(
            int(target.surface_token_count)
        )
    )

    group_key = (
        target.language,
        int(target.physical_line_count),
    )

    candidates = []

    for donor_family_id in (
        candidate_groups[group_key]
    ):
        donor = record_lookup[
            donor_family_id
        ]

        if (
            donor.repository
            == target.repository
        ):
            continue

        if (
            donor.exact_comment_hash
            == target.exact_comment_hash
        ):
            continue

        donor_surface_tokens = int(
            donor.surface_token_count
        )

        if not (
            lower_bound
            <= donor_surface_tokens
            <= upper_bound
        ):
            continue

        lexical_overlap = unigram_jaccard(
            target.lexical_token_set,
            donor.lexical_token_set,
        )

        if (
            lexical_overlap
            > SELECTED_TOPIC_JACCARD
        ):
            continue

        pair_token_count = (
            int(target.codebert_code_tokens)
            + int(
                donor.codebert_comment_tokens
            )
            + PAIR_SPECIAL_TOKENS
        )

        if (
            pair_token_count
            > CODEBERT_MAX_LENGTH
        ):
            continue

        candidates.append(
            {
                "donor_family_id": (
                    donor.family_id
                ),
                "donor_sample_id": (
                    donor.sample_id
                ),
                "donor_repository": (
                    donor.repository
                ),
                "donor_comment": (
                    donor.original_comment
                ),
                "donor_surface_tokens": (
                    donor_surface_tokens
                ),
                "surface_token_difference": (
                    donor_surface_tokens
                    - int(
                        target.surface_token_count
                    )
                ),
                "unigram_jaccard": float(
                    lexical_overlap
                ),
                "pair_token_count": int(
                    pair_token_count
                ),
                "selection_rank": stable_rank(
                    GENERATION_SEED,
                    target.family_id,
                    donor.family_id,
                ),
            }
        )

    candidates.sort(
        key=lambda candidate: (
            abs(
                candidate[
                    "surface_token_difference"
                ]
            ),
            candidate["unigram_jaccard"],
            candidate["selection_rank"],
        )
    )

    topic_candidate_lists[
        target.family_id
    ] = candidates


topic_assignments = {}
donor_use_counts = Counter()

target_assignment_order = sorted(
    generation_frame[
        "family_id"
    ],
    key=lambda family_id: (
        len(
            topic_candidate_lists[
                family_id
            ]
        ),
        stable_rank(
            GENERATION_SEED,
            family_id,
        ),
    ),
)

for family_id in target_assignment_order:
    candidates = topic_candidate_lists[
        family_id
    ]

    if not candidates:
        topic_assignments[
            family_id
        ] = None
        continue

    unused_candidates = [
        candidate
        for candidate in candidates
        if donor_use_counts[
            candidate["donor_family_id"]
        ] == 0
    ]

    selection_pool = (
        unused_candidates
        if unused_candidates
        else candidates
    )

    selected_candidate = min(
        selection_pool,
        key=lambda candidate: (
            donor_use_counts[
                candidate[
                    "donor_family_id"
                ]
            ],
            abs(
                candidate[
                    "surface_token_difference"
                ]
            ),
            candidate["unigram_jaccard"],
            candidate["selection_rank"],
        ),
    )

    donor_use_counts[
        selected_candidate[
            "donor_family_id"
        ]
    ] += 1

    topic_assignments[
        family_id
    ] = selected_candidate


shuffle_assignments = {}

for row in generation_frame.itertuples(
    index=False
):
    chunks, token_positions, tokens = (
        split_token_layout(
            row.original_comment
        )
    )

    candidate_permutations = (
        generate_candidate_permutations(
            tokens,
            row.family_id,
        )
    )

    candidates = []

    for candidate_tokens in (
        candidate_permutations
    ):
        shuffled_comment = (
            render_token_permutation(
                chunks,
                token_positions,
                candidate_tokens,
            )
        )

        if (
            shuffled_comment
            == row.original_comment
        ):
            continue

        bigram_overlap = (
            multiset_bigram_jaccard(
                tokens,
                candidate_tokens,
            )
        )

        if (
            bigram_overlap
            > SELECTED_SHUFFLE_JACCARD
        ):
            continue

        fixed_fraction = (
            fixed_position_fraction(
                tokens,
                candidate_tokens,
            )
        )

        candidates.append(
            {
                "comment": shuffled_comment,
                "bigram_jaccard": float(
                    bigram_overlap
                ),
                "fixed_position_fraction": (
                    float(fixed_fraction)
                ),
                "selection_rank": stable_rank(
                    GENERATION_SEED,
                    row.family_id,
                    "\x1f".join(
                        candidate_tokens
                    ),
                ),
            }
        )

    candidates.sort(
        key=lambda candidate: (
            candidate["bigram_jaccard"],
            candidate[
                "fixed_position_fraction"
            ],
            candidate["selection_rank"],
        )
    )

    shuffle_assignments[
        row.family_id
    ] = (
        candidates[0]
        if candidates
        else None
    )


generation_rows = []

for row in generation_frame.itertuples(
    index=False
):
    topic_assignment = (
        topic_assignments[
            row.family_id
        ]
    )

    shuffle_assignment = (
        shuffle_assignments[
            row.family_id
        ]
    )

    generation_rows.append(
        {
            "sample_id": row.sample_id,
            "family_id": row.family_id,
            "language": row.language,
            "repository": row.repository,
            "original_code": (
                row.original_code
            ),
            "original_comment": (
                row.original_comment
            ),
            "original_comment_lines": int(
                row.physical_line_count
            ),
            "original_surface_tokens": int(
                row.surface_token_count
            ),
            "original_pair_tokens": int(
                row.reconstructed_original_pair_tokens
            ),
            "topic_swap_success": (
                topic_assignment is not None
            ),
            "topic_candidate_count": len(
                topic_candidate_lists[
                    row.family_id
                ]
            ),
            "topic_donor_family_id": (
                topic_assignment[
                    "donor_family_id"
                ]
                if topic_assignment
                else ""
            ),
            "topic_donor_sample_id": (
                topic_assignment[
                    "donor_sample_id"
                ]
                if topic_assignment
                else ""
            ),
            "topic_donor_repository": (
                topic_assignment[
                    "donor_repository"
                ]
                if topic_assignment
                else ""
            ),
            "topic_swapped_comment": (
                topic_assignment[
                    "donor_comment"
                ]
                if topic_assignment
                else ""
            ),
            "topic_surface_tokens": (
                topic_assignment[
                    "donor_surface_tokens"
                ]
                if topic_assignment
                else None
            ),
            "topic_surface_token_difference": (
                topic_assignment[
                    "surface_token_difference"
                ]
                if topic_assignment
                else None
            ),
            "topic_unigram_jaccard": (
                topic_assignment[
                    "unigram_jaccard"
                ]
                if topic_assignment
                else None
            ),
            "topic_pair_tokens": (
                topic_assignment[
                    "pair_token_count"
                ]
                if topic_assignment
                else None
            ),
            "grammar_shuffle_success": (
                shuffle_assignment
                is not None
            ),
            "grammar_shuffled_comment": (
                shuffle_assignment[
                    "comment"
                ]
                if shuffle_assignment
                else ""
            ),
            "grammar_bigram_jaccard": (
                shuffle_assignment[
                    "bigram_jaccard"
                ]
                if shuffle_assignment
                else None
            ),
            "grammar_fixed_position_fraction": (
                shuffle_assignment[
                    "fixed_position_fraction"
                ]
                if shuffle_assignment
                else None
            ),
        }
    )


generation_audit = pd.DataFrame(
    generation_rows
)

successful_shuffle_mask = (
    generation_audit[
        "grammar_shuffle_success"
    ]
)

shuffle_token_lengths = (
    batched_token_lengths(
        generation_audit.loc[
            successful_shuffle_mask,
            "grammar_shuffled_comment",
        ]
    )
)

generation_audit[
    "grammar_comment_tokens"
] = pd.NA

generation_audit.loc[
    successful_shuffle_mask,
    "grammar_comment_tokens",
] = shuffle_token_lengths

generation_audit[
    "grammar_pair_tokens"
] = pd.NA

generation_audit.loc[
    successful_shuffle_mask,
    "grammar_pair_tokens",
] = (
    generation_frame.loc[
        successful_shuffle_mask,
        "codebert_code_tokens",
    ].to_numpy()
    + pd.Series(
        shuffle_token_lengths
    ).to_numpy()
    + PAIR_SPECIAL_TOKENS
)


selected_donor_final_use = {
    donor_family_id: int(use_count)
    for donor_family_id, use_count
    in donor_use_counts.items()
}

generation_audit[
    "topic_donor_final_use_count"
] = generation_audit[
    "topic_donor_family_id"
].map(
    selected_donor_final_use
)

generation_audit[
    "topic_donor_final_use_count"
] = generation_audit[
    "topic_donor_final_use_count"
].fillna(0).astype(int)

generation_audit[
    "both_perturbations_success"
] = (
    generation_audit[
        "topic_swap_success"
    ]
    & generation_audit[
        "grammar_shuffle_success"
    ]
)


generation_summary = (
    generation_audit
    .groupby(
        "language",
        as_index=False,
    )
    .agg(
        records=(
            "family_id",
            "count",
        ),
        topic_swap_success=(
            "topic_swap_success",
            "sum",
        ),
        grammar_shuffle_success=(
            "grammar_shuffle_success",
            "sum",
        ),
        both_perturbations_success=(
            "both_perturbations_success",
            "sum",
        ),
        unique_topic_donors=(
            "topic_donor_family_id",
            lambda values: (
                values[
                    values != ""
                ].nunique()
            ),
        ),
        maximum_topic_donor_reuse=(
            "topic_donor_final_use_count",
            "max",
        ),
        maximum_topic_pair_tokens=(
            "topic_pair_tokens",
            "max",
        ),
        maximum_grammar_pair_tokens=(
            "grammar_pair_tokens",
            "max",
        ),
    )
)

for column in [
    "topic_swap_success",
    "grammar_shuffle_success",
    "both_perturbations_success",
]:
    generation_summary[
        f"{column}_percent"
    ] = (
        100
        * generation_summary[column]
        / generation_summary["records"]
    ).round(2)


generation_audit.to_csv(
    GENERATION_OUTPUT_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)

generation_audit.to_csv(
    GENERATION_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

generation_summary.to_csv(
    GENERATION_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)


generation_manifest = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_sample": str(
        SAMPLE_CSV_PATH
    ),
    "records": int(
        len(generation_audit)
    ),
    "topic_swap_rule": {
        "same_language": True,
        "different_repository": True,
        "exact_physical_lines": True,
        "surface_token_caliper": (
            topic_decision[
                "selected_caliper_rule"
            ]
        ),
        "maximum_unigram_jaccard": (
            SELECTED_TOPIC_JACCARD
        ),
        "codebert_pair_limit": (
            CODEBERT_MAX_LENGTH
        ),
        "selection_priority": [
            "unused donor",
            "smallest surface-token difference",
            "lowest unigram Jaccard",
            "deterministic hash",
        ],
    },
    "grammar_shuffle_rule": {
        "tokens_preserved": True,
        "whitespace_preserved": True,
        "physical_lines_preserved": True,
        "maximum_bigram_jaccard": (
            SELECTED_SHUFFLE_JACCARD
        ),
    },
    "output_files": {
        "generation_draft": str(
            GENERATION_OUTPUT_PATH
        ),
        "generation_audit": str(
            GENERATION_AUDIT_PATH
        ),
        "language_summary": str(
            GENERATION_SUMMARY_PATH
        ),
    },
}

with GENERATION_MANIFEST_PATH.open(
    mode="w",
    encoding="utf-8",
) as handle:
    json.dump(
        generation_manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


print("Perturbation generation summary")
display(
    generation_summary[
        [
            "language",
            "records",
            "topic_swap_success",
            "topic_swap_success_percent",
            "grammar_shuffle_success",
            "grammar_shuffle_success_percent",
            "both_perturbations_success",
            "both_perturbations_success_percent",
            "unique_topic_donors",
            "maximum_topic_donor_reuse",
            "maximum_topic_pair_tokens",
            "maximum_grammar_pair_tokens",
        ]
    ]
)

print("\nGenerated files")
print("Draft dataset:", GENERATION_OUTPUT_PATH)
print("Generation audit:", GENERATION_AUDIT_PATH)
print("Language summary:", GENERATION_SUMMARY_PATH)
print("Manifest:", GENERATION_MANIFEST_PATH)

C:\Users\HP\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Perturbation generation summary


,language,records,topic_swap_success,topic_swap_success_percent,grammar_shuffle_success,grammar_shuffle_success_percent,both_perturbations_success,both_perturbations_success_percent,unique_topic_donors,maximum_topic_donor_reuse,maximum_topic_pair_tokens,maximum_grammar_pair_tokens
0,go,1250,1215,97.20,1243,99.44,1209,96.72,1194,3,505.0,508
1,java,1250,1216,97.28,1180,94.40,1146,91.68,1185,2,511.0,513
2,javascript,1250,1207,96.56,1184,94.72,1141,91.28,1182,2,509.0,513
3,php,1250,1220,97.60,1152,92.16,1122,89.76,1153,2,510.0,512
4,python,1250,1224,97.92,1244,99.52,1218,97.44,1201,2,511.0,512
5,ruby,1250,1185,94.80,1198,95.84,1134,90.72,1161,2,509.0,509



Generated files
Draft dataset: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_Perturbation_Generation_Draft.csv
Generation audit: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\perturbation_generation_audit.csv
Language summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\perturbation_generation_by_language.csv
Manifest: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\perturbation_generation_manifest.json


In [10]:
VALIDATION_AUDIT_PATH = (
    EDA_DIR
    / "perturbation_validation_audit.csv"
)

VALIDATION_SUMMARY_PATH = (
    EDA_DIR
    / "perturbation_validation_by_language.csv"
)

REPAIRED_DRAFT_PATH = (
    OUTPUT_DIR
    / "02_Perturbation_Generation_Validated_Draft.csv"
)


def get_pair_token_count(
    code_token_count,
    comment,
):
    comment_tokens = len(
        tokenizer(
            str(comment),
            add_special_tokens=False,
            truncation=False,
        )["input_ids"]
    )

    return (
        int(code_token_count)
        + int(comment_tokens)
        + PAIR_SPECIAL_TOKENS
    )


def physical_lines(text):
    return max(
        len(str(text).splitlines()),
        1,
    )


invalid_shuffle_mask = (
    generation_audit[
        "grammar_shuffle_success"
    ]
    & (
        pd.to_numeric(
            generation_audit[
                "grammar_pair_tokens"
            ],
            errors="coerce",
        )
        > CODEBERT_MAX_LENGTH
    )
)

invalid_shuffle_families = set(
    generation_audit.loc[
        invalid_shuffle_mask,
        "family_id",
    ]
)

print(
    "Invalid shuffled pairs before repair:",
    len(invalid_shuffle_families),
)


repair_results = {}

for family_id in invalid_shuffle_families:
    source_row = generation_frame.loc[
        generation_frame[
            "family_id"
        ] == family_id
    ].iloc[0]

    chunks, token_positions, tokens = (
        split_token_layout(
            source_row[
                "original_comment"
            ]
        )
    )

    candidate_permutations = (
        generate_candidate_permutations(
            tokens,
            family_id,
        )
    )

    valid_candidates = []

    for candidate_tokens in (
        candidate_permutations
    ):
        shuffled_comment = (
            render_token_permutation(
                chunks,
                token_positions,
                candidate_tokens,
            )
        )

        if (
            shuffled_comment
            == source_row[
                "original_comment"
            ]
        ):
            continue

        bigram_overlap = (
            multiset_bigram_jaccard(
                tokens,
                candidate_tokens,
            )
        )

        if (
            bigram_overlap
            > SELECTED_SHUFFLE_JACCARD
        ):
            continue

        pair_tokens = get_pair_token_count(
            source_row[
                "codebert_code_tokens"
            ],
            shuffled_comment,
        )

        if (
            pair_tokens
            > CODEBERT_MAX_LENGTH
        ):
            continue

        fixed_fraction = (
            fixed_position_fraction(
                tokens,
                candidate_tokens,
            )
        )

        valid_candidates.append(
            {
                "comment": shuffled_comment,
                "bigram_jaccard": float(
                    bigram_overlap
                ),
                "fixed_position_fraction": float(
                    fixed_fraction
                ),
                "pair_tokens": int(
                    pair_tokens
                ),
                "selection_rank": stable_rank(
                    GENERATION_SEED,
                    family_id,
                    "\x1f".join(
                        candidate_tokens
                    ),
                ),
            }
        )

    valid_candidates.sort(
        key=lambda candidate: (
            candidate[
                "bigram_jaccard"
            ],
            candidate[
                "fixed_position_fraction"
            ],
            candidate[
                "pair_tokens"
            ],
            candidate[
                "selection_rank"
            ],
        )
    )

    repair_results[family_id] = (
        valid_candidates[0]
        if valid_candidates
        else None
    )


for family_id, repair in (
    repair_results.items()
):
    row_mask = (
        generation_audit[
            "family_id"
        ] == family_id
    )

    if repair is None:
        generation_audit.loc[
            row_mask,
            "grammar_shuffle_success",
        ] = False

        generation_audit.loc[
            row_mask,
            "grammar_shuffled_comment",
        ] = ""

        generation_audit.loc[
            row_mask,
            "grammar_bigram_jaccard",
        ] = pd.NA

        generation_audit.loc[
            row_mask,
            "grammar_fixed_position_fraction",
        ] = pd.NA

        generation_audit.loc[
            row_mask,
            "grammar_comment_tokens",
        ] = pd.NA

        generation_audit.loc[
            row_mask,
            "grammar_pair_tokens",
        ] = pd.NA

    else:
        generation_audit.loc[
            row_mask,
            "grammar_shuffled_comment",
        ] = repair["comment"]

        generation_audit.loc[
            row_mask,
            "grammar_bigram_jaccard",
        ] = repair[
            "bigram_jaccard"
        ]

        generation_audit.loc[
            row_mask,
            "grammar_fixed_position_fraction",
        ] = repair[
            "fixed_position_fraction"
        ]

        generation_audit.loc[
            row_mask,
            "grammar_pair_tokens",
        ] = repair[
            "pair_tokens"
        ]

        generation_audit.loc[
            row_mask,
            "grammar_comment_tokens",
        ] = (
            repair["pair_tokens"]
            - int(
                generation_frame.loc[
                    row_mask,
                    "codebert_code_tokens",
                ].iloc[0]
            )
            - PAIR_SPECIAL_TOKENS
        )


topic_success_mask = (
    generation_audit[
        "topic_swap_success"
    ]
)

shuffle_success_mask = (
    generation_audit[
        "grammar_shuffle_success"
    ]
)


generation_audit[
    "topic_lines_preserved"
] = False

generation_audit.loc[
    topic_success_mask,
    "topic_lines_preserved",
] = (
    generation_audit.loc[
        topic_success_mask,
        "topic_swapped_comment",
    ].map(
        physical_lines
    ).to_numpy()
    == generation_audit.loc[
        topic_success_mask,
        "original_comment_lines",
    ].to_numpy()
)


generation_audit[
    "topic_different_repository"
] = False

generation_audit.loc[
    topic_success_mask,
    "topic_different_repository",
] = (
    generation_audit.loc[
        topic_success_mask,
        "repository",
    ].to_numpy()
    != generation_audit.loc[
        topic_success_mask,
        "topic_donor_repository",
    ].to_numpy()
)


generation_audit[
    "topic_jaccard_valid"
] = False

generation_audit.loc[
    topic_success_mask,
    "topic_jaccard_valid",
] = (
    pd.to_numeric(
        generation_audit.loc[
            topic_success_mask,
            "topic_unigram_jaccard",
        ],
        errors="raise",
    )
    <= SELECTED_TOPIC_JACCARD
)


generation_audit[
    "topic_pair_length_valid"
] = False

generation_audit.loc[
    topic_success_mask,
    "topic_pair_length_valid",
] = (
    pd.to_numeric(
        generation_audit.loc[
            topic_success_mask,
            "topic_pair_tokens",
        ],
        errors="raise",
    )
    <= CODEBERT_MAX_LENGTH
)


generation_audit[
    "shuffle_lines_preserved"
] = False

generation_audit.loc[
    shuffle_success_mask,
    "shuffle_lines_preserved",
] = (
    generation_audit.loc[
        shuffle_success_mask,
        "grammar_shuffled_comment",
    ].map(
        physical_lines
    ).to_numpy()
    == generation_audit.loc[
        shuffle_success_mask,
        "original_comment_lines",
    ].to_numpy()
)


generation_audit[
    "shuffle_surface_tokens_preserved"
] = False

for index in generation_audit.index[
    shuffle_success_mask
]:
    original_tokens = surface_tokens(
        generation_audit.at[
            index,
            "original_comment",
        ]
    )

    shuffled_tokens = surface_tokens(
        generation_audit.at[
            index,
            "grammar_shuffled_comment",
        ]
    )

    generation_audit.at[
        index,
        "shuffle_surface_tokens_preserved",
    ] = (
        Counter(original_tokens)
        == Counter(shuffled_tokens)
    )


generation_audit[
    "shuffle_whitespace_preserved"
] = False

for index in generation_audit.index[
    shuffle_success_mask
]:
    original_whitespace = re.findall(
        r"\s+",
        generation_audit.at[
            index,
            "original_comment",
        ],
        flags=re.UNICODE,
    )

    shuffled_whitespace = re.findall(
        r"\s+",
        generation_audit.at[
            index,
            "grammar_shuffled_comment",
        ],
        flags=re.UNICODE,
    )

    generation_audit.at[
        index,
        "shuffle_whitespace_preserved",
    ] = (
        original_whitespace
        == shuffled_whitespace
    )


generation_audit[
    "shuffle_changed"
] = False

generation_audit.loc[
    shuffle_success_mask,
    "shuffle_changed",
] = (
    generation_audit.loc[
        shuffle_success_mask,
        "grammar_shuffled_comment",
    ].to_numpy()
    != generation_audit.loc[
        shuffle_success_mask,
        "original_comment",
    ].to_numpy()
)


generation_audit[
    "shuffle_jaccard_valid"
] = False

generation_audit.loc[
    shuffle_success_mask,
    "shuffle_jaccard_valid",
] = (
    pd.to_numeric(
        generation_audit.loc[
            shuffle_success_mask,
            "grammar_bigram_jaccard",
        ],
        errors="raise",
    )
    <= SELECTED_SHUFFLE_JACCARD
)


generation_audit[
    "shuffle_pair_length_valid"
] = False

generation_audit.loc[
    shuffle_success_mask,
    "shuffle_pair_length_valid",
] = (
    pd.to_numeric(
        generation_audit.loc[
            shuffle_success_mask,
            "grammar_pair_tokens",
        ],
        errors="raise",
    )
    <= CODEBERT_MAX_LENGTH
)


generation_audit[
    "topic_valid"
] = (
    generation_audit[
        "topic_swap_success"
    ]
    & generation_audit[
        "topic_lines_preserved"
    ]
    & generation_audit[
        "topic_different_repository"
    ]
    & generation_audit[
        "topic_jaccard_valid"
    ]
    & generation_audit[
        "topic_pair_length_valid"
    ]
)


generation_audit[
    "shuffle_valid"
] = (
    generation_audit[
        "grammar_shuffle_success"
    ]
    & generation_audit[
        "shuffle_lines_preserved"
    ]
    & generation_audit[
        "shuffle_surface_tokens_preserved"
    ]
    & generation_audit[
        "shuffle_whitespace_preserved"
    ]
    & generation_audit[
        "shuffle_changed"
    ]
    & generation_audit[
        "shuffle_jaccard_valid"
    ]
    & generation_audit[
        "shuffle_pair_length_valid"
    ]
)


generation_audit[
    "both_perturbations_valid"
] = (
    generation_audit[
        "topic_valid"
    ]
    & generation_audit[
        "shuffle_valid"
    ]
)


validation_summary = (
    generation_audit
    .groupby(
        "language",
        as_index=False,
    )
    .agg(
        records=(
            "family_id",
            "count",
        ),
        valid_topic_swaps=(
            "topic_valid",
            "sum",
        ),
        valid_grammar_shuffles=(
            "shuffle_valid",
            "sum",
        ),
        valid_complete_triplets=(
            "both_perturbations_valid",
            "sum",
        ),
        maximum_topic_pair_tokens=(
            "topic_pair_tokens",
            "max",
        ),
        maximum_shuffle_pair_tokens=(
            "grammar_pair_tokens",
            "max",
        ),
        maximum_topic_donor_reuse=(
            "topic_donor_final_use_count",
            "max",
        ),
    )
)


for column in [
    "valid_topic_swaps",
    "valid_grammar_shuffles",
    "valid_complete_triplets",
]:
    validation_summary[
        f"{column}_percent"
    ] = (
        100
        * validation_summary[column]
        / validation_summary["records"]
    ).round(2)


assert (
    generation_audit.loc[
        generation_audit[
            "topic_swap_success"
        ],
        [
            "topic_lines_preserved",
            "topic_different_repository",
            "topic_jaccard_valid",
            "topic_pair_length_valid",
        ],
    ].all(axis=1)
).all()

assert (
    generation_audit.loc[
        generation_audit[
            "grammar_shuffle_success"
        ],
        [
            "shuffle_lines_preserved",
            "shuffle_surface_tokens_preserved",
            "shuffle_whitespace_preserved",
            "shuffle_changed",
            "shuffle_jaccard_valid",
            "shuffle_pair_length_valid",
        ],
    ].all(axis=1)
).all()

assert (
    pd.to_numeric(
        generation_audit.loc[
            generation_audit[
                "topic_valid"
            ],
            "topic_pair_tokens",
        ],
        errors="raise",
    )
    <= CODEBERT_MAX_LENGTH
).all()

assert (
    pd.to_numeric(
        generation_audit.loc[
            generation_audit[
                "shuffle_valid"
            ],
            "grammar_pair_tokens",
        ],
        errors="raise",
    )
    <= CODEBERT_MAX_LENGTH
).all()


generation_audit.to_csv(
    REPAIRED_DRAFT_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)

generation_audit.to_csv(
    VALIDATION_AUDIT_PATH,
    index=False,
    encoding="utf-8",
)

validation_summary.to_csv(
    VALIDATION_SUMMARY_PATH,
    index=False,
    encoding="utf-8",
)


print(
    "Shuffled pairs repaired:",
    sum(
        repair is not None
        for repair in repair_results.values()
    ),
)

print(
    "Shuffled pairs removed after repair failure:",
    sum(
        repair is None
        for repair in repair_results.values()
    ),
)

print("\nPerturbation validation summary")
display(
    validation_summary[
        [
            "language",
            "records",
            "valid_topic_swaps",
            "valid_topic_swaps_percent",
            "valid_grammar_shuffles",
            "valid_grammar_shuffles_percent",
            "valid_complete_triplets",
            "valid_complete_triplets_percent",
            "maximum_topic_pair_tokens",
            "maximum_shuffle_pair_tokens",
            "maximum_topic_donor_reuse",
        ]
    ]
)

print("\nSaved validation outputs")
print("Validated draft:", REPAIRED_DRAFT_PATH)
print("Validation audit:", VALIDATION_AUDIT_PATH)
print("Validation summary:", VALIDATION_SUMMARY_PATH)

Invalid shuffled pairs before repair: 2
Shuffled pairs repaired: 2
Shuffled pairs removed after repair failure: 0

Perturbation validation summary


,language,records,valid_topic_swaps,valid_topic_swaps_percent,valid_grammar_shuffles,valid_grammar_shuffles_percent,valid_complete_triplets,valid_complete_triplets_percent,maximum_topic_pair_tokens,maximum_shuffle_pair_tokens,maximum_topic_donor_reuse
0,go,1250,1215,97.20,1243,99.44,1209,96.72,505.0,508,3
1,java,1250,1216,97.28,1180,94.40,1146,91.68,511.0,512,2
2,javascript,1250,1207,96.56,1184,94.72,1141,91.28,509.0,512,2
3,php,1250,1220,97.60,1152,92.16,1122,89.76,510.0,512,2
4,python,1250,1224,97.92,1244,99.52,1218,97.44,511.0,512,2
5,ruby,1250,1185,94.80,1198,95.84,1134,90.72,509.0,509,2



Saved validation outputs
Validated draft: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_Perturbation_Generation_Validated_Draft.csv
Validation audit: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\perturbation_validation_audit.csv
Validation summary: C:\Users\HP\Desktop\thesis_preprocessing\outputs\rework_v2\02_distractor_generation\perturbation_validation_by_language.csv


In [12]:
import csv
import hashlib
import json
import math
import sqlite3
from datetime import datetime, timezone

import pandas as pd


FINAL_OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed_rework_v2"
)

FINAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

ALL_VALID_LONG_PATH = (
    FINAL_OUTPUT_DIR
    / "02_All_Valid_Conditions_Long.csv"
)

RQ1_PATH = (
    FINAL_OUTPUT_DIR
    / "02_RQ1_Complete_Triplets_Long.csv"
)

RQ2_PATH = (
    FINAL_OUTPUT_DIR
    / "02_RQ2_Balanced_Complete_Triplets_Long.csv"
)

RQ3_PATH = (
    FINAL_OUTPUT_DIR
    / "02_RQ3_Original_Methods.csv"
)

MANUAL_REVIEW_PATH = (
    FINAL_OUTPUT_DIR
    / "02_Blinded_Manual_Review.csv"
)

MANUAL_REVIEW_KEY_PATH = (
    FINAL_OUTPUT_DIR
    / "02_Blinded_Manual_Review_Key.csv"
)

FINAL_MANIFEST_PATH = (
    FINAL_OUTPUT_DIR
    / "02_Distractor_Generation_Final_Manifest.json"
)

BALANCED_RQ2_SIZE_PER_LANGUAGE = int(
    validation_summary[
        "valid_complete_triplets"
    ].min()
)

MANUAL_REVIEW_PER_CONDITION_LANGUAGE = 10
MANUAL_REVIEW_SEED = 20260805
Z_95 = 1.96


def deterministic_hash(*parts):
    value = "\x1f".join(
        str(part)
        for part in parts
    )

    return int.from_bytes(
        hashlib.sha256(
            value.encode("utf-8")
        ).digest()[:8],
        byteorder="big",
        signed=False,
    )


metadata_columns = [
    "sample_id",
    "family_id",
    "language",
    "repository",
    "sample_order",
    "within_repository_rank",
    "original_split",
    "file_path",
    "function_name",
    "commit_sha",
    "source_url",
    "shard_relative_path",
    "shard_line_number",
    "code_token_count",
    "comment_token_count",
    "combined_token_count",
    "code_line_count",
    "comment_line_count",
    "exact_code_hash",
    "exact_comment_hash",
    "exact_pair_hash",
]

metadata_columns = [
    column
    for column in metadata_columns
    if column in sample.columns
]

sample_metadata = sample[
    metadata_columns
].copy()

audit_with_metadata = (
    generation_audit.merge(
        sample_metadata,
        on=[
            "sample_id",
            "family_id",
            "language",
            "repository",
        ],
        how="left",
        validate="one_to_one",
    )
)


def build_condition_frame(
    source,
    condition,
    comment_column,
    pair_token_column,
):
    condition_frame = source.copy()

    condition_frame[
        "condition"
    ] = condition

    condition_frame[
        "code"
    ] = condition_frame[
        "original_code"
    ]

    condition_frame[
        "comment"
    ] = condition_frame[
        comment_column
    ]

    condition_frame[
        "pair_token_count"
    ] = pd.to_numeric(
        condition_frame[
            pair_token_column
        ],
        errors="raise",
    ).astype(int)

    condition_frame[
        "donor_family_id"
    ] = ""

    condition_frame[
        "donor_sample_id"
    ] = ""

    condition_frame[
        "donor_repository"
    ] = ""

    condition_frame[
        "unigram_jaccard"
    ] = pd.NA

    condition_frame[
        "bigram_jaccard"
    ] = pd.NA

    condition_frame[
        "fixed_position_fraction"
    ] = pd.NA

    if condition == "topic_swap":
        condition_frame[
            "donor_family_id"
        ] = condition_frame[
            "topic_donor_family_id"
        ]

        condition_frame[
            "donor_sample_id"
        ] = condition_frame[
            "topic_donor_sample_id"
        ]

        condition_frame[
            "donor_repository"
        ] = condition_frame[
            "topic_donor_repository"
        ]

        condition_frame[
            "unigram_jaccard"
        ] = condition_frame[
            "topic_unigram_jaccard"
        ]

    if condition == "grammar_shuffle":
        condition_frame[
            "bigram_jaccard"
        ] = condition_frame[
            "grammar_bigram_jaccard"
        ]

        condition_frame[
            "fixed_position_fraction"
        ] = condition_frame[
            "grammar_fixed_position_fraction"
        ]

    return condition_frame


original_rows = build_condition_frame(
    audit_with_metadata,
    condition="original",
    comment_column="original_comment",
    pair_token_column="original_pair_tokens",
)

topic_rows = build_condition_frame(
    audit_with_metadata.loc[
        audit_with_metadata[
            "topic_valid"
        ]
    ].copy(),
    condition="topic_swap",
    comment_column="topic_swapped_comment",
    pair_token_column="topic_pair_tokens",
)

shuffle_rows = build_condition_frame(
    audit_with_metadata.loc[
        audit_with_metadata[
            "shuffle_valid"
        ]
    ].copy(),
    condition="grammar_shuffle",
    comment_column="grammar_shuffled_comment",
    pair_token_column="grammar_pair_tokens",
)


long_export_columns = (
    metadata_columns
    + [
        "condition",
        "code",
        "comment",
        "pair_token_count",
        "donor_family_id",
        "donor_sample_id",
        "donor_repository",
        "unigram_jaccard",
        "bigram_jaccard",
        "fixed_position_fraction",
    ]
)

long_export_columns = list(
    dict.fromkeys(
        long_export_columns
    )
)

all_valid_long = (
    pd.concat(
        [
            original_rows[
                long_export_columns
            ],
            topic_rows[
                long_export_columns
            ],
            shuffle_rows[
                long_export_columns
            ],
        ],
        ignore_index=True,
    )
    .sort_values(
        [
            "language",
            "sample_id",
            "condition",
        ]
    )
    .reset_index(drop=True)
)


complete_family_table = (
    audit_with_metadata.loc[
        audit_with_metadata[
            "both_perturbations_valid"
        ],
        [
            "sample_id",
            "family_id",
            "language",
            "repository",
        ],
    ]
    .copy()
)

complete_family_ids = set(
    complete_family_table[
        "family_id"
    ]
)

rq1_long = (
    all_valid_long.loc[
        all_valid_long[
            "family_id"
        ].isin(
            complete_family_ids
        )
    ]
    .copy()
    .sort_values(
        [
            "language",
            "sample_id",
            "condition",
        ]
    )
    .reset_index(drop=True)
)


complete_family_table[
    "repository_hash"
] = complete_family_table.apply(
    lambda row: deterministic_hash(
        MANUAL_REVIEW_SEED,
        row["language"],
        row["repository"],
    ),
    axis=1,
)

complete_family_table[
    "method_hash"
] = complete_family_table[
    "family_id"
].map(
    lambda family_id: deterministic_hash(
        MANUAL_REVIEW_SEED,
        family_id,
    )
)

complete_family_table = (
    complete_family_table
    .sort_values(
        [
            "language",
            "repository",
            "method_hash",
            "family_id",
        ]
    )
    .reset_index(drop=True)
)

complete_family_table[
    "within_repository_rank"
] = (
    complete_family_table
    .groupby(
        [
            "language",
            "repository",
        ]
    )
    .cumcount()
    + 1
)

balanced_rq2_families = (
    complete_family_table
    .sort_values(
        [
            "language",
            "within_repository_rank",
            "repository_hash",
            "method_hash",
            "family_id",
        ]
    )
    .groupby(
        "language",
        group_keys=False,
    )
    .head(
        BALANCED_RQ2_SIZE_PER_LANGUAGE
    )
    .copy()
)

balanced_rq2_family_ids = set(
    balanced_rq2_families[
        "family_id"
    ]
)

rq2_long = (
    rq1_long.loc[
        rq1_long[
            "family_id"
        ].isin(
            balanced_rq2_family_ids
        )
    ]
    .copy()
    .sort_values(
        [
            "language",
            "sample_id",
            "condition",
        ]
    )
    .reset_index(drop=True)
)


rq3_original = sample.copy()

rq3_original[
    "condition"
] = "original"

rq3_original[
    "code"
] = rq3_original[
    "original_code"
]

rq3_original[
    "comment"
] = rq3_original[
    "original_comment"
]


assert (
    rq1_long.groupby(
        "family_id"
    )["condition"].nunique()
    == 3
).all()

assert (
    rq2_long.groupby(
        "family_id"
    )["condition"].nunique()
    == 3
).all()

assert (
    balanced_rq2_families[
        "language"
    ].value_counts()
    .reindex(EXPECTED_LANGUAGES)
    == BALANCED_RQ2_SIZE_PER_LANGUAGE
).all()

assert (
    rq3_original[
        "language"
    ].value_counts()
    .reindex(EXPECTED_LANGUAGES)
    == 1250
).all()

assert (
    all_valid_long[
        "pair_token_count"
    ] <= 512
).all()

assert all_valid_long[
    "code"
].str.strip().ne("").all()

assert all_valid_long[
    "comment"
].str.strip().ne("").all()


condition_order = [
    "topic_swap",
    "grammar_shuffle",
    "original",
]

manual_review_parts = []

for language_value in EXPECTED_LANGUAGES:
    used_family_ids = set()

    for condition_value in condition_order:
        candidates = (
            all_valid_long.loc[
                (
                    all_valid_long[
                        "language"
                    ] == language_value
                )
                & (
                    all_valid_long[
                        "condition"
                    ] == condition_value
                )
                & (
                    ~all_valid_long[
                        "family_id"
                    ].isin(
                        used_family_ids
                    )
                )
            ]
            .copy()
        )

        candidates[
            "review_selection_hash"
        ] = candidates[
            "family_id"
        ].map(
            lambda family_id: deterministic_hash(
                MANUAL_REVIEW_SEED,
                "manual-review",
                language_value,
                condition_value,
                family_id,
            )
        )

        selected_review_rows = (
            candidates
            .sort_values(
                [
                    "review_selection_hash",
                    "family_id",
                ]
            )
            .head(
                MANUAL_REVIEW_PER_CONDITION_LANGUAGE
            )
            .copy()
        )

        assert len(
            selected_review_rows
        ) == (
            MANUAL_REVIEW_PER_CONDITION_LANGUAGE
        )

        used_family_ids.update(
            selected_review_rows[
                "family_id"
            ]
        )

        manual_review_parts.append(
            selected_review_rows
        )


manual_review_key = (
    pd.concat(
        manual_review_parts,
        ignore_index=True,
    )
    .copy()
)

manual_review_key[
    "review_order_hash"
] = manual_review_key.apply(
    lambda row: deterministic_hash(
        MANUAL_REVIEW_SEED,
        "review-order",
        row["language"],
        row["condition"],
        row["family_id"],
    ),
    axis=1,
)

manual_review_key = (
    manual_review_key
    .sort_values(
        [
            "review_order_hash",
            "family_id",
        ]
    )
    .reset_index(drop=True)
)

manual_review_key[
    "blind_id"
] = [
    f"REVIEW-{index:04d}"
    for index in range(
        1,
        len(manual_review_key) + 1,
    )
]


manual_review_blinded = (
    manual_review_key[
        [
            "blind_id",
            "language",
            "code",
            "comment",
        ]
    ]
    .copy()
)

manual_review_blinded[
    "semantic_consistency_rating_1_to_5"
] = ""

manual_review_blinded[
    "grammaticality_rating_1_to_5"
] = ""

manual_review_blinded[
    "reviewer_confidence_1_to_5"
] = ""

manual_review_blinded[
    "reviewer_notes"
] = ""


manual_review_key_export = (
    manual_review_key[
        [
            "blind_id",
            "sample_id",
            "family_id",
            "language",
            "repository",
            "condition",
        ]
    ]
    .copy()
)


connection = sqlite3.connect(
    CANDIDATE_DB_PATH,
    timeout=120,
)

try:
    eligible_populations = pd.read_sql_query(
        """
        SELECT
            language,
            COUNT(*) AS eligible_test_population
        FROM eligible_candidate_records
        WHERE original_split = 'test'
        GROUP BY language
        """,
        connection,
    )

finally:
    connection.close()


rq1_counts = (
    complete_family_table
    .groupby(
        "language"
    )
    .size()
    .rename(
        "rq1_complete_families"
    )
)

rq2_counts = (
    balanced_rq2_families
    .groupby(
        "language"
    )
    .size()
    .rename(
        "rq2_balanced_families"
    )
)

rq3_counts = (
    rq3_original
    .groupby(
        "language"
    )
    .size()
    .rename(
        "rq3_original_families"
    )
)

design_validation = (
    eligible_populations
    .set_index("language")
    .join(rq1_counts)
    .join(rq2_counts)
    .join(rq3_counts)
    .reset_index()
)


def finite_population_margin_percent(
    population_size,
    sample_size,
):
    return (
        Z_95
        * math.sqrt(
            0.25
            / sample_size
            * (
                population_size
                - sample_size
            )
            / (
                population_size
                - 1
            )
        )
        * 100
    )


for sample_column, prefix in [
    (
        "rq1_complete_families",
        "rq1",
    ),
    (
        "rq2_balanced_families",
        "rq2",
    ),
    (
        "rq3_original_families",
        "rq3",
    ),
]:
    design_validation[
        f"{prefix}_worst_case_95_moe_percent"
    ] = design_validation.apply(
        lambda row: round(
            finite_population_margin_percent(
                int(
                    row[
                        "eligible_test_population"
                    ]
                ),
                int(
                    row[
                        sample_column
                    ]
                ),
            ),
            3,
        ),
        axis=1,
    )


assert (
    design_validation[
        [
            "rq1_worst_case_95_moe_percent",
            "rq2_worst_case_95_moe_percent",
            "rq3_worst_case_95_moe_percent",
        ]
    ]
    <= 3.0
).all().all()


all_valid_long.to_csv(
    ALL_VALID_LONG_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)

rq1_long.to_csv(
    RQ1_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)

rq2_long.to_csv(
    RQ2_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)

rq3_original.to_csv(
    RQ3_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)

manual_review_blinded.to_csv(
    MANUAL_REVIEW_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)

manual_review_key_export.to_csv(
    MANUAL_REVIEW_KEY_PATH,
    index=False,
    encoding="utf-8",
    quoting=csv.QUOTE_ALL,
    lineterminator="\n",
)


final_manifest = {
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_test_sample": str(
        SAMPLE_CSV_PATH
    ),
    "source_validated_generation": str(
        REPAIRED_DRAFT_PATH
    ),
    "languages": list(
        EXPECTED_LANGUAGES
    ),
    "topic_swap_rule": {
        "same_language": True,
        "different_repository": True,
        "exact_physical_line_count": True,
        "surface_token_caliper": (
            "plus or minus 10 percent, "
            "with a minimum tolerance of 2"
        ),
        "maximum_unigram_jaccard": 0.15,
        "maximum_codebert_pair_tokens": 512,
    },
    "grammar_shuffle_rule": {
        "surface_tokens_preserved": True,
        "whitespace_preserved": True,
        "physical_lines_preserved": True,
        "maximum_multiset_bigram_jaccard": 0.05,
        "maximum_codebert_pair_tokens": 512,
    },
    "rq1": {
        "design": (
            "All families with valid original, "
            "topic-swap and grammar-shuffle conditions"
        ),
        "allocation": (
            "All available complete triplets; "
            "paired analysis within each family"
        ),
        "families": int(
            complete_family_table[
                "family_id"
            ].nunique()
        ),
        "rows": int(
            len(rq1_long)
        ),
        "output": str(
            RQ1_PATH
        ),
    },
    "rq2": {
        "design": (
            "Repository-aware balanced subset "
            "of complete triplets"
        ),
        "families_per_language": int(
            BALANCED_RQ2_SIZE_PER_LANGUAGE
        ),
        "total_families": int(
            balanced_rq2_families[
                "family_id"
            ].nunique()
        ),
        "rows": int(
            len(rq2_long)
        ),
        "output": str(
            RQ2_PATH
        ),
    },
    "rq3": {
        "design": (
            "All original methods from the "
            "balanced master test sample"
        ),
        "families_per_language": 1250,
        "total_families": int(
            rq3_original[
                "family_id"
            ].nunique()
        ),
        "output": str(
            RQ3_PATH
        ),
    },
    "precision_requirement": {
        "confidence_level_percent": 95,
        "worst_case_proportion": 0.5,
        "maximum_margin_of_error_percent": 3.0,
        "finite_population_correction": True,
        "all_rq_datasets_pass": True,
    },
    "manual_review": {
        "blinded": True,
        "items_per_condition_per_language": (
            MANUAL_REVIEW_PER_CONDITION_LANGUAGE
        ),
        "total_items": int(
            len(manual_review_blinded)
        ),
        "families_disjoint_between_conditions_within_language": True,
        "review_file": str(
            MANUAL_REVIEW_PATH
        ),
        "key_file": str(
            MANUAL_REVIEW_KEY_PATH
        ),
    },
    "output_files": {
        "all_valid_conditions": str(
            ALL_VALID_LONG_PATH
        ),
        "rq1": str(
            RQ1_PATH
        ),
        "rq2": str(
            RQ2_PATH
        ),
        "rq3": str(
            RQ3_PATH
        ),
        "manual_review": str(
            MANUAL_REVIEW_PATH
        ),
        "manual_review_key": str(
            MANUAL_REVIEW_KEY_PATH
        ),
        "manifest": str(
            FINAL_MANIFEST_PATH
        ),
    },
}

with FINAL_MANIFEST_PATH.open(
    mode="w",
    encoding="utf-8",
) as handle:
    json.dump(
        final_manifest,
        handle,
        indent=2,
        ensure_ascii=False,
    )


dataset_summary = pd.DataFrame(
    {
        "dataset": [
            "All valid conditions",
            "RQ1 complete triplets",
            "RQ2 balanced triplets",
            "RQ3 original methods",
            "Blinded manual review",
        ],
        "families_or_items": [
            all_valid_long[
                "family_id"
            ].nunique(),
            rq1_long[
                "family_id"
            ].nunique(),
            rq2_long[
                "family_id"
            ].nunique(),
            rq3_original[
                "family_id"
            ].nunique(),
            len(
                manual_review_blinded
            ),
        ],
        "rows": [
            len(
                all_valid_long
            ),
            len(
                rq1_long
            ),
            len(
                rq2_long
            ),
            len(
                rq3_original
            ),
            len(
                manual_review_blinded
            ),
        ],
    }
)


print("Final dataset summary")
display(dataset_summary)

print("\nRQ-specific precision validation")
display(design_validation)

print("\nFinal files")
print("All valid conditions:", ALL_VALID_LONG_PATH)
print("RQ1 dataset:", RQ1_PATH)
print("RQ2 dataset:", RQ2_PATH)
print("RQ3 dataset:", RQ3_PATH)
print("Manual review:", MANUAL_REVIEW_PATH)
print("Manual review key:", MANUAL_REVIEW_KEY_PATH)
print("Final manifest:", FINAL_MANIFEST_PATH)


C:\Users\HP\AppData\Local\Temp\ipykernel_20836\2222791945.py:283: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(


Final dataset summary


,dataset,families_or_items,rows
0,All valid conditions,7500,21968
1,RQ1 complete triplets,6970,20910
2,RQ2 balanced triplets,6732,20196
3,RQ3 original methods,7500,7500
4,Blinded manual review,180,180



RQ-specific precision validation


,language,eligible_test_population,rq1_complete_families,rq2_balanced_families,rq3_original_families,rq1_worst_case_95_moe_percent,rq2_worst_case_95_moe_percent,rq3_worst_case_95_moe_percent
0,go,13159,1209,1122,1250,2.686,2.798,2.637
1,java,22412,1146,1122,1250,2.820,2.852,2.694
2,javascript,5059,1141,1122,1250,2.553,2.581,2.405
3,php,23251,1122,1122,1250,2.854,2.854,2.696
4,python,14180,1218,1122,1250,2.685,2.808,2.647
5,ruby,1990,1134,1122,1250,1.909,1.933,1.691



Final files
All valid conditions: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_All_Valid_Conditions_Long.csv
RQ1 dataset: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_RQ1_Complete_Triplets_Long.csv
RQ2 dataset: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_RQ2_Balanced_Complete_Triplets_Long.csv
RQ3 dataset: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_RQ3_Original_Methods.csv
Manual review: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_Blinded_Manual_Review.csv
Manual review key: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_Blinded_Manual_Review_Key.csv
Final manifest: C:\Users\HP\Desktop\thesis_preprocessing\data\processed_rework_v2\02_Distractor_Generation_Final_Manifest.json
